In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import os
import json
import hashlib
import zipfile
from pathlib import Path
from datetime import datetime
from IPython.display import display, FileLink

# Kaggle workspace
WORK_DIR = Path("/kaggle/working")
ZIP_PATH = WORK_DIR / "CAFRNet_v2_COMPLETE_BACKUP.zip"

assert WORK_DIR.exists(), "Kaggle working directory not found!"

# Exclude credentials, Git internals and temporary caches
EXCLUDED_DIRS = {
    ".git", ".kaggle", ".config",
    "__pycache__", ".cache"
}

manifest = []
files_to_backup = []

for root, dirs, files in os.walk(WORK_DIR):
    dirs[:] = [d for d in dirs if d not in EXCLUDED_DIRS]

    for filename in files:
        path = Path(root) / filename

        if path == ZIP_PATH or not path.is_file():
            continue

        relative_path = path.relative_to(WORK_DIR)

        # Skip the generated backup itself
        if filename.endswith(".pyc"):
            continue

        files_to_backup.append(path)

print("Files found:", len(files_to_backup))

# Generate checksums and backup manifest
for path in files_to_backup:
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    manifest.append({
        "file": str(path.relative_to(WORK_DIR)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256.hexdigest()
    })

manifest_data = {
    "project": "CAFR-Net v2 Reviewer Experiments",
    "created": datetime.now().isoformat(),
    "file_count": len(manifest),
    "files": manifest
}

# Create ZIP
with zipfile.ZipFile(
    ZIP_PATH, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as z:
    for path in files_to_backup:
        z.write(path, path.relative_to(WORK_DIR))

    z.writestr(
        "BACKUP_MANIFEST.json",
        json.dumps(manifest_data, indent=2)
    )

print("\nBACKUP COMPLETED!")
print("ZIP:", ZIP_PATH)
print("Size (GB):", round(ZIP_PATH.stat().st_size / (1024**3), 3))
print("Files included:", len(files_to_backup))

# Display downloadable link
display(FileLink(str(ZIP_PATH)))


In [ ]:

import os
import subprocess
from pathlib import Path
import torch

REPO = Path("/kaggle/working/cartoonization-robustness-study")
BRANCH = "research/cafr-net-phase-iv"
GITHUB = "https://github.com/Karthik6-cpu/cartoonization-robustness-study.git"

# Clone or update repository
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", GITHUB, str(REPO)],
        check=True
    )
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)

SOURCE = REPO / "research/CAFRNet_v2/source"

print("\nGPU:")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU NOT DETECTED")

print("\nRESEARCH SOURCE:")
print(SOURCE)
print("Training script:", (SOURCE / "train_architecture.py").exists())
print("Matrix launcher:", (SOURCE / "run_reviewer_matrix.sh").exists())

print("\nKAGGLE INPUT DATASETS:")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\nDATASET ROOT CANDIDATES:")
for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
    for train in base.rglob("train"):
        if train.is_dir() and (train.parent / "val").is_dir():
            print("DATA ROOT:", train.parent)

print("\nEXISTING EXPERIMENT CHECKPOINTS:")
for p in Path("/kaggle/working").rglob("latest_model.pt"):
    print(p)

print("\nSETUP CHECK COMPLETE")


In [ ]:

from pathlib import Path

for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\nSearching: {base}")
    for p in base.rglob("train"):
        if p.is_dir() and (p.parent / "val").is_dir():
            train_classes = list(p.iterdir())
            val_classes = list((p.parent / "val").iterdir())
            print("DATA ROOT:", p.parent)
            print("Training classes:", len(train_classes))
            print("Validation classes:", len(val_classes))


In [ ]:

import requests
import zipfile
from pathlib import Path

RELEASE_TAG = "cafrnet-v2-checkpoints-2026-11-25"
REPO = "Karthik6-cpu/cartoonization-robustness-study"

WORK = Path("/kaggle/working")
ZIP_PATH = WORK / "cafr_data_2026-11-25.zip"
EXTRACT_DIR = WORK / "dataset_extracted"

# Find the uploaded dataset in the GitHub release
api = f"https://api.github.com/repos/{REPO}/releases/tags/{RELEASE_TAG}"
response = requests.get(api, timeout=60)
response.raise_for_status()

assets = response.json()["assets"]
asset = next(
    (a for a in assets if a["name"] == "cafr_data_2026-11-25.zip"),
    None
)

if asset is None:
    raise RuntimeError("Dataset ZIP not found in the GitHub release.")

print("Dataset:", asset["name"])
print("Size (GB):", round(asset["size"] / 1024**3, 2))

# Download with progress
with requests.get(asset["browser_download_url"], stream=True, timeout=120) as r:
    r.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                f.write(chunk)

print("Download completed:", ZIP_PATH)

# Extract
EXTRACT_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)

# Locate the actual ImageFolder root
matches = [
    p.parent for p in EXTRACT_DIR.rglob("train")
    if p.is_dir() and (p.parent / "val").is_dir()
]

if not matches:
    raise RuntimeError("Could not find a folder containing train/ and val/.")

DATA_ROOT = matches[0]
print("\nDATA_ROOT =", DATA_ROOT)
print("Train classes:", len(list((DATA_ROOT / "train").iterdir())))
print("Validation classes:", len(list((DATA_ROOT / "val").iterdir())))

print("\nDATASET READY")


In [ ]:
# import os
import sys
import time
import json
import zipfile
import subprocess
from pathlib import Path
from datetime import datetime
from IPython.display import display, FileLink

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------
WORK = Path("/kaggle/working")
REPO = WORK / "cartoonization-robustness-study"
SOURCE = REPO / "research/CAFRNet_v2/source"

DATA_ROOT = WORK / "dataset_extracted/cafr_data"
OUT_ROOT = WORK / "cafr_reviewer_runs"
RUN_ROOT = OUT_ROOT / "imagenette"

LOG = WORK / "CAFRNet_Imagenette_Reviewer_Training.log"
ZIP_PATH = WORK / "CAFRNet_Imagenette_Reviewer_Complete.zip"

# --------------------------------------------------
# PREFLIGHT CHECKS
# --------------------------------------------------
import torch
import torchvision
import cv2
import numpy as np

assert torch.cuda.is_available(), "CUDA GPU not available!"
assert (SOURCE / "train_architecture.py").exists()
assert (SOURCE / "run_reviewer_matrix.sh").exists()
assert (DATA_ROOT / "train").is_dir(), "Training folder missing!"
assert (DATA_ROOT / "val").is_dir(), "Validation folder missing!"

train_classes = sorted(p.name for p in (DATA_ROOT / "train").iterdir() if p.is_dir())
val_classes = sorted(p.name for p in (DATA_ROOT / "val").iterdir() if p.is_dir())

assert train_classes == val_classes, "Train/validation class mismatch!"
assert len(train_classes) == 10, "Unexpected number of classes!"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Classes:", len(train_classes))
print("Dataset:", DATA_ROOT)

# --------------------------------------------------
# CREATE OUTPUT DIRECTORY
# --------------------------------------------------
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# RUN / RESUME THE CONTROLLED MATRIX
# --------------------------------------------------
env = os.environ.copy()
env.update({
    "EPOCHS": "20",
    "BATCH_SIZE": "32",
    "WORKERS": "2",
    "RESUME": "1"
})

command = [
    "bash",
    str(SOURCE / "run_reviewer_matrix.sh"),
    "imagenette",
    str(DATA_ROOT),
    str(OUT_ROOT)
]

print("\nStarting/resuming 18 Imagenette experiments...")
print("Output:", RUN_ROOT)
print("Log:", LOG)

with open(LOG, "a", buffering=1) as log:
    log.write("\n\n===== SESSION START " + datetime.now().isoformat() + " =====\n")

    process = subprocess.Popen(
        command,
        cwd=SOURCE,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Experiment launcher stopped with exit code {return_code}. "
        "Check the log and rerun this cell to resume from saved checkpoints."
    )

print("\nALL 18 IMAGENETTE TRAINING RUNS COMPLETED.")

# --------------------------------------------------
# VERIFY CHECKPOINTS
# --------------------------------------------------
run_dirs = sorted(p for p in RUN_ROOT.iterdir() if p.is_dir())

checkpoint_report = []
for folder in run_dirs:
    best = folder / "best_model.pt"
    latest = folder / "latest_model.pt"

    checkpoint_report.append({
        "run": folder.name,
        "best_checkpoint": best.exists(),
        "latest_checkpoint": latest.exists()
    })

print("\nCHECKPOINT REPORT:")
for item in checkpoint_report:
    print(item)

if len(run_dirs) != 18 or not all(
    x["best_checkpoint"] and x["latest_checkpoint"]
    for x in checkpoint_report
):
    raise RuntimeError(
        "Expected 18 runs with both checkpoints. "
        "Inspect the report before treating the matrix as complete."
    )

# --------------------------------------------------
# CREATE FINAL RESEARCH BACKUP
# --------------------------------------------------
backup_manifest = {
    "project": "CAFR-Net v2 Imagenette Reviewer Experiments",
    "created": datetime.now().isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "epochs": 20,
    "seeds": [42, 123, 2026],
    "runs": checkpoint_report
}

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as archive:

    for base in [RUN_ROOT]:
        for file in base.rglob("*"):
            if file.is_file():
                archive.write(file, file.relative_to(WORK))

    archive.write(LOG, LOG.name)

    # Preserve the exact source used for the experiments
    for file in SOURCE.glob("*.py"):
        archive.write(file, Path("research_source") / file.name)

    archive.writestr(
        "BACKUP_MANIFEST.json",
        json.dumps(backup_manifest, indent=2)
    )

print("\nBACKUP CREATED SUCCESSFULLY")
print("File:", ZIP_PATH)
print("Size (GB):", round(ZIP_PATH.stat().st_size / 1024**3, 3))

display(FileLink(str(ZIP_PATH)))


In [ ]:
from pathlib import Path
import os

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

print("Searching for CAFR-Net checkpoints...")

found = []

for root in search_roots:
    if root.exists():
        for p in root.rglob("latest_model.pt"):
            found.append(p)
            print("FOUND:", p)
            print("SIZE MB:", round(p.stat().st_size / 1024**2, 2))

if not found:
    print("No latest_model.pt checkpoint found.")

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

extensions = {".pt", ".pth", ".ckpt", ".zip", ".tar", ".gz"}

print("Searching saved research files...")

for root in roots:
    if not root.exists():
        continue

    for p in root.rglob("*"):
        if p.is_file() and (
            p.suffix.lower() in extensions
            or p.name in ["history.json", "run_metadata.json"]
        ):
            print(f"{p} | {p.stat().st_size / (1024**2):.2f} MB")

In [ ]:
!ls -lah /kaggle/working
!ls -lah /kaggle/input

In [ ]:
import os, subprocess
from pathlib import Path

REPO = "https://github.com/Karthik6-cpu/cartoonization-robustness-study.git"
WORK = Path("/kaggle/working")

subprocess.run([
    "git", "clone", "--branch", "research/cafr-net-phase-iv",
    REPO, str(WORK / "cartoonization-robustness-study")
], check=True)

print("Research source restored.")

In [ ]:
import os, zipfile, urllib.request
from pathlib import Path

url = "https://codeload.github.com/Karthik6-cpu/cartoonization-robustness-study/zip/refs/heads/research/cafr-net-phase-iv"
archive = "/kaggle/working/research_source.zip"

print("Downloading research source...")
urllib.request.urlretrieve(url, archive)

with zipfile.ZipFile(archive) as z:
    z.extractall("/kaggle/working")

print("Extraction completed.")

In [ ]:
import socket

for host in ["github.com", "codeload.github.com"]:
    try:
        print(host, socket.gethostbyname(host))
    except Exception as e:
        print(host, "FAILED:", e)

In [ ]:
import requests

url = "https://github.com"
try:
    r = requests.get(url, timeout=15)
    print("Internet:", r.status_code)
except Exception as e:
    print("Internet unavailable:", e)

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("Available Kaggle datasets:")
for p in root.iterdir():
    print("\n", p.name)

print("\nSearching for model checkpoints:")
for p in root.rglob("*"):
    if p.is_file() and (
        p.name in ["latest_model.pt", "best_model.pt"]
        or p.suffix.lower() == ".zip"
    ):
        print(p, round(p.stat().st_size / 1024**2, 2), "MB")

In [ ]:
import torch
from pathlib import Path

p = Path(
    "/kaggle/input/datasets/maddulaadithya/seed42/"
    "cafrnet_seed42/latest_model.pt"
)

ck = torch.load(p, map_location="cpu", weights_only=False)

print("CHECKPOINT DETAILS")
print("=" * 40)

for key in ["epoch", "step", "seed", "architecture",
            "arch", "best_accuracy", "val_accuracy"]:
    if key in ck:
        print(f"{key}: {ck[key]}")

print("\nAvailable keys:", list(ck.keys()))

In [ ]:
import pandas as pd
import json
from pathlib import Path

out = Path("/kaggle/working/CAFRNet_Seed42_Experiment_Logs")
out.mkdir(parents=True, exist_ok=True)

records = [
    (1, 0.4240265705842768, 0.012516850491268374, 0.9989506820566632),
    (2, 0.1986316203049171, 0.0109680111491652, 0.9989506820566632),
    (3, 0.13291285761985044, 0.008469299272172826, 0.9989506820566632),
    (4, 0.07741360756196344, 0.008561153203341328, 0.9989506820566632),
    (5, 0.05509207118083861, 0.00903586882270698, 0.9979013641133263),
    (6, 0.04670948188746687, 0.00593447021521669, 0.9989506820566632),
    (7, 0.04318638281356902, 0.00931532095512693, 0.9979013641133263),
    (8, 0.03675928292336698, 0.020825889618176302, 0.9958027282266527),
    (9, 0.0239239439959675, 0.018900346703391465, 0.9958027282266527),
    (10, 0.024206070189166092, 0.01126683643845498, 0.9968520461699895),
    (11, 0.014815680340576362, 0.015690098728531453, 0.9968520461699895),
    (12, 0.018061510430270093, 0.010710120002594773, 0.9968520461699895)
]

df = pd.DataFrame(
    records,
    columns=["epoch", "train_loss", "val_loss", "val_accuracy"]
)

df.insert(0, "seed", 42)
df.insert(1, "architecture", "cafrnet_v2")

df.to_csv(out / "seed42_epochs_1_to_12.csv", index=False)

metadata = {
    "architecture": "cafrnet_v2",
    "seed": 42,
    "completed_epochs": 12,
    "next_planned_epoch": 13,
    "checkpoint_status": "November 26 checkpoint unavailable",
    "metrics": df.to_dict(orient="records")
}

with open(out / "seed42_epochs_1_to_12.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(df.to_string(index=False))
print("\nSaved CSV and JSON:", out)

In [ ]:
from IPython.display import display, FileLink

folder = "/kaggle/working/CAFRNet_Seed42_Experiment_Logs"

display(FileLink(folder + "/seed42_epochs_1_to_12.csv"))
display(FileLink(folder + "/seed42_epochs_1_to_12.json"))

In [ ]:
from pathlib import Path
import os

folder = Path("/kaggle/working/CAFRNet_Seed42_Experiment_Logs")
zip_path = Path("/kaggle/working/CAFRNet_Seed42_Epoch1_to_12_Backup.zip")

print("CSV:", (folder / "seed42_epochs_1_to_12.csv").exists())
print("JSON:", (folder / "seed42_epochs_1_to_12.json").exists())
print("ZIP:", zip_path.exists())

print("\nBackup files:")
for p in Path("/kaggle/working").glob("CAFRNet_Seed42*"):
    print(p, round(p.stat().st_size / 1024, 2), "KB")

In [ ]:
from pathlib import Path
import torch, os, json, shutil, time

CKPT = Path("/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42/latest_model.pt")

assert CKPT.exists(), f"Checkpoint not found: {CKPT}"

ck = torch.load(CKPT, map_location="cpu", weights_only=False)

print("Checkpoint verified!")
print("Architecture:", ck.get("architecture"))
print("Seed:", ck.get("seed"))
print("Recovered epoch:", ck.get("epoch"))
print("Best accuracy:", ck.get("best_accuracy"))
print("Available keys:", list(ck.keys()))

assert ck.get("architecture") == "cafrnet_v2"
assert ck.get("seed") == 42
assert ck.get("epoch") == 20

In [ ]:
from pathlib import Path
import shutil, json, time

BASE = Path("/kaggle/working/CAFRNet_Seed42_Finetune_From_Epoch13")
CHECKPOINT_DIR = BASE / "checkpoints"
BACKUP_DIR = BASE / "backup"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(CKPT, BACKUP_DIR / "initial_recovered_epoch20.pt")

metadata = {
    "experiment": "CAFRNet_v2_seed42_finetune",
    "display_start_epoch": 13,
    "initialized_from": "Recovered older CAFR-Net v2 seed42 epoch20 checkpoint",
    "optimizer": "Fresh optimizer",
    "scheduler": "Fresh scheduler",
    "previous_november26_epochs_1_to_12": "Metrics preserved separately; exact checkpoint unavailable",
    "created": time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(BASE / "experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Experiment folders ready:", BASE)
print("Initial checkpoint backed up.")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/cartoonization-robustness-study")

print("Repository exists:", ROOT.exists())

if ROOT.exists():
    print("Training files:")
    for p in ROOT.rglob("train_architecture.py"):
        print(p)
    for p in ROOT.rglob("run_reviewer_matrix.sh"):
        print(p)

In [ ]:
import requests

try:
    r = requests.get(
        "https://github.com",
        timeout=15
    )
    print("Internet available:", r.status_code)
except Exception as e:
    print("Internet unavailable:", repr(e))

In [ ]:
from pathlib import Path

print("Kaggle input datasets:")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\nSearching for CAFR-Net training files...")
for name in ["run_reviewer_matrix.sh", "train_architecture.py"]:
    matches = list(Path("/kaggle/input").rglob(name))
    print(name, ":", matches)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input/datasets").rglob("*"):
    if p.is_file():
        print(p)

In [ ]:
import torch, json
from pathlib import Path

CKPT = "/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42/latest_model.pt"

ck = torch.load(CKPT, map_location="cpu", weights_only=False)

print("Architecture:", ck["architecture"])
print("Seed:", ck["seed"])
print("Original epoch:", ck["epoch"])
print("Model config:")
print(json.dumps(ck["model_config"], indent=2, default=str))
print("Classes:", ck["classes"])
print("Metadata:", json.dumps(ck.get("metadata", {}), indent=2, default=str))

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\n--- {root} ---")
    for p in root.iterdir():
        print(p)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file() and (
        p.name in ["run_reviewer_matrix.sh", "train_architecture.py"]
        or p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ):
        print(p)

In [ ]:
from pathlib import Path
import json, shutil

BASE = Path("/kaggle/working/CAFRNet_Seed42_Fresh_Run")
BASE.mkdir(parents=True, exist_ok=True)

# Preserve the recovered checkpoint separately
OLD = Path("/kaggle/working/CAFRNet_Seed42_Finetune_From_Epoch13")
if (OLD / "backup/initial_recovered_epoch20.pt").exists():
    print("Recovered Epoch 20 checkpoint preserved.")

config = {
    "architecture": "cafrnet_v2",
    "seed": 42,
    "start_epoch": 1,
    "total_epochs": 20,
    "batch_size": 16,
    "image_size": 224,
    "optimizer": "AdamW",
    "resume": False,
    "checkpoint_every_epoch": True
}

with open(BASE / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Fresh experiment folder ready:", BASE)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("train_architecture.py"):
    print("TRAINER:", p)

for p in Path("/kaggle/input").rglob("run_reviewer_matrix.sh"):
    print("RUN SCRIPT:", p)

In [ ]:
from pathlib import Path
import torch

print("=== AVAILABLE DATASETS ===")
for p in Path("/kaggle/input").iterdir():
    print(p)

print("\n=== SEARCHING FOR CHECKPOINTS AND DATA ===")
for p in Path("/kaggle/input").rglob("*"):
    if p.is_file() and (
        p.name.endswith((".pt", ".pth")) or
        p.name in ("train", "val")
    ):
        print(p)

ckpt = Path(
    "/kaggle/input/datasets/maddulaadithya/seed42/"
    "cafrnet_seed42/latest_model.pt"
)

if ckpt.exists():
    data = torch.load(ckpt, map_location="cpu", weights_only=False)
    print("\nCheckpoint epoch:", data.get("epoch"))
    print("Architecture:", data.get("architecture"))
    print("Seed:", data.get("seed"))

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_dir() and p.name in ["train", "val"]:
        print(p)

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320"

TRAIN_DIR = DATA_ROOT + "/train"
VAL_DIR   = DATA_ROOT + "/val"

In [ ]:
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320"
)

for split in ["train", "val"]:
    folder = DATA_ROOT / split
    classes = sorted(p.name for p in folder.iterdir() if p.is_dir())
    images = sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    )

    print(f"\n{split.upper()}")
    print("Classes:", len(classes))
    print("Images:", images)
    print("Class names:", classes)

In [ ]:
import torch
from pathlib import Path

ckpt_path = Path(
    "/kaggle/input/datasets/maddulaadithya/seed42/"
    "cafrnet_seed42/latest_model.pt"
)

ck = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False
)

print("Architecture:", ck.get("architecture"))
print("Checkpoint epoch:", ck.get("epoch"))
print("Seed:", ck.get("seed"))
print("Checkpoint classes:", ck.get("classes"))
print("Model config:", ck.get("model_config"))
print("Best accuracy:", ck.get("best_accuracy"))

print("\nCheckpoint metadata:")
print(ck.get("metadata"))

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\nSearching: {root}")
    for name in ["cafr_net.py", "train_architecture.py"]:
        matches = list(root.rglob(name))
        for p in matches:
            print("FOUND:", p)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(
    accept=".py",
    multiple=True,
    description="Upload Python files"
)

display(uploader)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*.py"):
    print("FOUND:", p)

In [ ]:
from pathlib import Path

print("=== KAGGLE INPUT DATASETS ===")

for p in Path("/kaggle/input").iterdir():
    print("\nDataset:", p.name)

    for f in p.rglob("*"):
        if f.is_file():
            print("  ", f.name, "->", f)

print("\n=== WORKING DIRECTORY ===")
for p in Path("/kaggle/working").iterdir():
    print(p.name)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)
        

In [ ]:
from pathlib import Path

# Check original split manifests
seed_dir = Path(
    "/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42"
)

print("=== ORIGINAL SPLIT FILES ===")
for name in ["train_split_manifest.csv",
             "validation_split_manifest.csv",
             "run_metadata.json"]:
    p = seed_dir / name
    print(name, "FOUND" if p.exists() else "MISSING")

# Check source Python files
print("\n=== CAFR-NET SOURCE FILES ===")
for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    for name in ["cafr_net.py", "train_architecture.py"]:
        matches = list(root.rglob(name))
        for p in matches:
            print("FOUND:", p)

# Check checkpoints
print("\n=== CHECKPOINTS ===")
for name in ["latest_model.pt", "best_model.pt"]:
    matches = list(Path("/kaggle/input").rglob(name))
    for p in matches:
        print("FOUND:", p)

In [ ]:
import pandas as pd
from pathlib import Path

seed_dir = Path(
    "/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42"
)

for filename in [
    "train_split_manifest.csv",
    "validation_split_manifest.csv"
]:
    path = seed_dir / filename
    df = pd.read_csv(path)

    print(f"\n===== {filename} =====")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import json
import torch

DATA_ROOT = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320"
)

SEED_DIR = Path(
    "/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42"
)

train_df = pd.read_csv(SEED_DIR / "train_split_manifest.csv")
val_df = pd.read_csv(SEED_DIR / "validation_split_manifest.csv")

print("=== MANIFEST FILE VERIFICATION ===")

for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df)
]:
    found = sum(
        (DATA_ROOT / "train" / p).exists()
        for p in df["relative_path"]
    )
    print(f"{name}: {found}/{len(df)} files found in train folder")

print("\n=== CHECKPOINT METADATA ===")

ckpt = torch.load(
    SEED_DIR / "latest_model.pt",
    map_location="cpu",
    weights_only=False
)

print("Epoch:", ckpt.get("epoch"))
print("Architecture:", ckpt.get("architecture"))
print("Best accuracy:", ckpt.get("best_accuracy"))

print("\n=== RUN METADATA ===")

with open(SEED_DIR / "run_metadata.json") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2)[:6000])

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working")
]

for root in roots:
    print(f"\nSearching {root}")
    for p in root.rglob("*.py"):
        print(p)

In [ ]:
from pathlib import Path

for name in ["cafr_net.py", "train_architecture.py"]:
    matches = list(Path("/kaggle/input").rglob(name))
    print(f"\n{name}:")
    for p in matches:
        print(p)

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\n========== {root} ==========")
    for p in root.rglob("*"):
        if p.is_file():
            print(p)

In [ ]:
from pathlib import Path
import pypdf

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")

for p in ROOT.rglob("*.pdf"):
    print("\nFILE:", p.name)
    reader = pypdf.PdfReader(str(p))
    content = "\n".join(page.extract_text() or "" for page in reader.pages)
    print("Pages:", len(reader.pages))
    print("Extracted characters:", len(content))
    print("Preview:\n", content[:1500])

In [ ]:
from pathlib import Path
import ast
import pypdf

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")
OUT = Path("/kaggle/working/CAFRNet_source")
OUT.mkdir(parents=True, exist_ok=True)

files = {
    "cafr_net.py": next(ROOT.rglob("*cafr_net.py*.pdf")),
    "train_architecture.py": next(ROOT.rglob("*train_architecture.py*.pdf")),
}

for output_name, pdf_path in files.items():
    reader = pypdf.PdfReader(str(pdf_path))
    source = "\n".join(page.extract_text() or "" for page in reader.pages)

    # Validate Python syntax before saving
    try:
        ast.parse(source)
    except SyntaxError as e:
        print(f"\n❌ {output_name}: Syntax error")
        print(f"Line {e.lineno}: {e.msg}")
        continue

    output_path = OUT / output_name
    output_path.write_text(source, encoding="utf-8")

    print(f"✅ Extracted: {output_path}")
    print(f"   Characters: {len(source):,}")
    print(f"   Syntax: VALID")

print("\nExtracted files:")
for p in OUT.glob("*.py"):
    print(p.name, "-", p.stat().st_size, "bytes")

In [ ]:
from pathlib import Path
import pypdf

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")

pdfs = {
    "cafr_net.py": next(ROOT.rglob("*cafr_net.py*.pdf")),
    "train_architecture.py": next(ROOT.rglob("*train_architecture.py*.pdf")),
}

for name, pdf_path in pdfs.items():
    reader = pypdf.PdfReader(str(pdf_path))
    source = "\n".join(page.extract_text() or "" for page in reader.pages)
    lines = source.splitlines()

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    for i in range(44, min(65, len(lines))):
        print(f"{i+1}: {lines[i]!r}")

In [ ]:
from pathlib import Path
import pypdf
import re
import ast

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")
OUT = Path("/kaggle/working/CAFRNet_source")
OUT.mkdir(parents=True, exist_ok=True)

files = {
    "cafr_net.py": next(ROOT.rglob("*cafr_net.py*.pdf")),
    "train_architecture.py": next(ROOT.rglob("*train_architecture.py*.pdf")),
}

for output_name, pdf_path in files.items():
    reader = pypdf.PdfReader(str(pdf_path))
    source = "\n".join(page.extract_text() or "" for page in reader.pages)

    cleaned_lines = []

    for line in source.splitlines():
        # Remove PDF webpage headers and footers
        if "raw.githubusercontent.com" in line:
            continue
        if re.match(r"^\s*\d{2}/\d{2}/\d{4},\s+\d{2}:\d{2}", line):
            continue

        cleaned_lines.append(line)

    cleaned_source = "\n".join(cleaned_lines).strip() + "\n"

    try:
        ast.parse(cleaned_source)
        output_path = OUT / output_name
        output_path.write_text(cleaned_source, encoding="utf-8")

        print(f"✅ {output_name}: VALID Python syntax")
        print(f"   Saved: {output_path}")
        print(f"   Lines: {len(cleaned_lines)}")

    except SyntaxError as e:
        print(f"❌ {output_name}: Syntax error")
        print(f"   Line {e.lineno}: {e.msg}")
        print("   Problem:", cleaned_lines[e.lineno - 1] if e.lineno else "Unknown")

print("\nFinal source directory:")
for p in OUT.glob("*.py"):
    print(p.name, f"({p.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")
pdf_path = next(ROOT.rglob("*train_architecture.py*.pdf"))

import pypdf
reader = pypdf.PdfReader(str(pdf_path))
source = "\n".join(page.extract_text() or "" for page in reader.pages)

import re

lines = []
for line in source.splitlines():
    if "raw.githubusercontent.com" in line:
        continue
    if re.match(r"^\s*\d{2}/\d{2}/\d{4},\s+\d{2}:\d{2}", line):
        continue
    lines.append(line)

print("TRAINING SCRIPT — LINES 130–165")
print("=" * 65)

for i in range(129, min(165, len(lines))):
    print(f"{i+1}: {lines[i]!r}")

In [ ]:
from pathlib import Path
import pypdf
import re
import ast

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")
OUT = Path("/kaggle/working/CAFRNet_source")
OUT.mkdir(parents=True, exist_ok=True)

pdf_path = next(ROOT.rglob("*train_architecture.py*.pdf"))

reader = pypdf.PdfReader(str(pdf_path))
source = "\n".join(page.extract_text() or "" for page in reader.pages)

cleaned_lines = []

for line in source.splitlines():
    if "raw.githubusercontent.com" in line:
        continue
    if re.match(r"^\s*\d{2}/\d{2}/\d{4},\s+\d{2}:\d{2}", line):
        continue
    cleaned_lines.append(line)

source = "\n".join(cleaned_lines)

# Repair the PDF-induced line break
source = source.replace(
    'help="Root containing train/ and val/ ImageFolder \ndirectories")',
    'help="Root containing train/ and val/ ImageFolder directories")'
)

# Validate and save
ast.parse(source)

output_path = OUT / "train_architecture.py"
output_path.write_text(source + "\n", encoding="utf-8")

print("✅ train_architecture.py: VALID Python syntax")
print("Saved:", output_path)
print("Lines:", len(source.splitlines()))

# Verify both source files
for filename in ["cafr_net.py", "train_architecture.py"]:
    path = OUT / filename
    ast.parse(path.read_text(encoding="utf-8"))
    print("✅ Verified:", filename)

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
lines = p.read_text(encoding="utf-8").splitlines()

print("TRAINING SCRIPT — FINAL 35 LINES")
print("=" * 65)

for i in range(max(0, len(lines)-35), len(lines)):
    print(f"{i+1}: {lines[i]!r}")

In [ ]:
from pathlib import Path
import pypdf
import re

ROOT = Path("/kaggle/input/datasets/maddulaadithya/dataset")
OUT = Path("/kaggle/working/CAFRNet_source")
OUT.mkdir(parents=True, exist_ok=True)

pdf = next(ROOT.rglob("*train_architecture.py*.pdf"))
reader = pypdf.PdfReader(str(pdf))

source = "\n".join(page.extract_text() or "" for page in reader.pages)

# Remove webpage headers and PDF footers
lines = []
for line in source.splitlines():
    if "raw.githubusercontent.com" in line or "githubusercontent.com" in line:
        continue
    if re.match(r"^\s*\d{2}/\d{2}/\d{4},\s+\d{2}:\d{2}", line):
        continue
    lines.append(line)

# Repair the split help string
source = "\n".join(lines)
source = source.replace(
    'help="Root containing train/ and val/ ImageFolder \ndirectories")',
    'help="Root containing train/ and val/ ImageFolder directories")'
)

# Save the recovered script BEFORE validation
p = OUT / "train_architecture.py"
p.write_text(source + "\n", encoding="utf-8")

print("Recovered file:", p)
print("Total lines:", len(source.splitlines()))
print("\nFINAL 45 LINES:")
for i, line in enumerate(source.splitlines()[-45:], 1):
    print(f"{len(source.splitlines())-45+i}: {line!r}")

In [ ]:
from pathlib import Path
import ast

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
lines = p.read_text(encoding="utf-8").splitlines()

print("RESUME SECTION — LINES 215–275")
print("=" * 65)

for i in range(214, min(275, len(lines))):
    print(f"{i+1}: {lines[i]!r}")

print("\n" + "=" * 65)
try:
    ast.parse("\n".join(lines))
    print("SYNTAX: VALID")
except SyntaxError as e:
    print("SYNTAX ERROR:", e)

In [ ]:
from pathlib import Path
import ast

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
source = p.read_text(encoding="utf-8")

start = source.index('        if "loader_generator_state" in ck:')
end = source.index('        if start_epoch>=args.freeze_epochs:', start)

corrected_block = '''        if "loader_generator_state" in ck:
            generator.set_state(
                ck["loader_generator_state"].detach().cpu().to(dtype=torch.uint8)
            )
        if "python_rng_state" in ck:
            random.setstate(ck["python_rng_state"])
        if "numpy_rng_state" in ck:
            np.random.set_state(ck["numpy_rng_state"])
        if "torch_rng_state" in ck:
            torch.set_rng_state(
                ck["torch_rng_state"].detach().cpu().to(dtype=torch.uint8)
            )
        if torch.cuda.is_available() and ck.get("cuda_rng_state"):
            torch.cuda.set_rng_state_all([
                s.detach().cpu().to(dtype=torch.uint8)
                for s in ck["cuda_rng_state"]
            ])
'''

source = source[:start] + corrected_block + source[end:]

# Validate before saving
ast.parse(source)

p.write_text(source, encoding="utf-8")

print("✅ Training script syntax: VALID")
print("✅ Checkpoint restoration block repaired")
print("✅ Script saved:", p)

# Verify both source files
for name in ["cafr_net.py", "train_architecture.py"]:
    file = p.parent / name
    ast.parse(file.read_text(encoding="utf-8"))
    print("✅", name, "validated")

In [ ]:
from pathlib import Path
import torch
import json
import pandas as pd

BASE = Path("/kaggle/input/datasets/maddulaadithya")
CKPT_DIR = BASE / "seed42/cafrnet_seed42"
DATASET_ROOT = Path("/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320")
SOURCE = Path("/kaggle/working/CAFRNet_source")

print("SOURCE FILES")
for name in ["cafr_net.py", "train_architecture.py"]:
    p = SOURCE / name
    print(name, "EXISTS:", p.exists())

print("\nCHECKPOINT VERIFICATION")
for name in ["latest_model.pt", "best_model.pt"]:
    p = CKPT_DIR / name
    print(name, "EXISTS:", p.exists())

ck = torch.load(CKPT_DIR / "latest_model.pt",
                map_location="cpu", weights_only=False)

print("\nCheckpoint architecture:", ck.get("architecture"))
print("Seed:", ck.get("seed"))
print("Saved epoch:", ck.get("epoch"))
print("Best accuracy:", ck.get("best_accuracy"))
print("Model config:", ck.get("model_config"))
print("Classes:", len(ck.get("classes", [])))

print("\nORIGINAL SPLIT")
for name in ["train_split_manifest.csv", "validation_split_manifest.csv"]:
    p = CKPT_DIR / name
    if p.exists():
        df = pd.read_csv(p)
        print(name, "rows:", len(df), "columns:", list(df.columns))
    else:
        print(name, "MISSING")

print("\nIMAGENETTE")
print("Train directory:", (DATASET_ROOT / "train").exists())
print("Validation directory:", (DATASET_ROOT / "val").exists())

print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

BASE = Path("/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42")
IMAGE_ROOT = Path("/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320/train")

WORK = Path("/kaggle/working/CAFRNet_continuation")
DATA = WORK / "cafr_data"
OUT = WORK / "cafrnet_seed42_continuation"

for split in ["train", "val"]:
    manifest_name = (
        "train_split_manifest.csv"
        if split == "train"
        else "validation_split_manifest.csv"
    )
    df = pd.read_csv(BASE / manifest_name)

    missing = []
    for row in df.itertuples(index=False):
        source = IMAGE_ROOT / row.relative_path
        if not source.is_file():
            missing.append(str(source))
            continue

        destination = DATA / split / row.class_name / Path(row.relative_path).name
        destination.parent.mkdir(parents=True, exist_ok=True)

        if not destination.exists():
            destination.symlink_to(source)

    print(f"{split}: {len(df)} manifest entries")
    print(f"{split}: {len(missing)} missing images")

    if missing:
        print("First missing paths:", missing[:5])
        raise RuntimeError("Dataset reconstruction failed.")

# Copy checkpoint files into a separate writable output folder
OUT.mkdir(parents=True, exist_ok=True)

for name in ["latest_model.pt", "best_model.pt"]:
    shutil.copy2(BASE / name, OUT / name)
    print("Checkpoint copied:", name)

print("\nTRAIN:", sum(1 for _ in (DATA / "train").rglob("*") if _.is_file()))
print("VAL:", sum(1 for _ in (DATA / "val").rglob("*") if _.is_file()))
print("DATA:", DATA)
print("OUTPUT:", OUT)

In [ ]:
from pathlib import Path
import torch
import subprocess
import sys

SOURCE = Path("/kaggle/working/CAFRNet_source")
OUT = Path("/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation")

ck = torch.load(
    OUT / "latest_model.pt",
    map_location="cpu",
    weights_only=False
)

required = [
    "model", "optimizer", "scheduler", "epoch",
    "architecture", "seed"
]

print("CHECKPOINT RESUME REQUIREMENTS")
for key in required:
    print(f"{'✅' if key in ck else '❌'} {key}")

print("\nCHECKPOINT DETAILS")
print("Epoch:", ck.get("epoch"))
print("Architecture:", ck.get("architecture"))
print("Seed:", ck.get("seed"))
print("Optimizer state:", "optimizer" in ck)
print("Scheduler state:", "scheduler" in ck)
print("Scaler state:", "scaler" in ck)
print("History entries:", len(ck.get("history", [])))

print("\nTRAINING SCRIPT ARGUMENTS")
result = subprocess.run(
    [sys.executable, str(SOURCE / "train_architecture.py"), "--help"],
    capture_output=True,
    text=True,
    cwd=str(SOURCE)
)
print(result.stdout)

if result.returncode != 0:
    print("SCRIPT ERROR:")
    print(result.stderr[-3000:])
else:
    print("✅ Training script imports and CLI successfully")

print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable")

In [ ]:
import subprocess
import sys
from pathlib import Path

SOURCE = Path("/kaggle/working/CAFRNet_source")
SCRIPT = SOURCE / "train_architecture.py"

result = subprocess.run(
    [sys.executable, str(SCRIPT), "--help"],
    cwd=str(SOURCE),
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("HELP OUTPUT:")
print(result.stdout[:5000] if result.stdout else "(No help output)")
print("ERROR OUTPUT:")
print(result.stderr[-3000:] if result.stderr else "(No errors)")

In [ ]:
import subprocess
import sys
from pathlib import Path

SOURCE = Path("/kaggle/working/CAFRNet_source")
WORK = Path("/kaggle/working/CAFRNet_continuation")

SCRIPT = SOURCE / "train_architecture.py"
DATA = WORK / "cafr_data"
OUT = WORK / "cafrnet_seed42_continuation"

cmd = [
    sys.executable, "-u", str(SCRIPT),
    "--arch", "cafrnet_v2",
    "--data", str(DATA),
    "--out", str(OUT),
    "--seed", "42",
    "--epochs", "30",
    "--batch-size", "16",
    "--workers", "2",
    "--image-size", "224",
    "--lambda-consistency", "0.25",
    "--encoder-lr", "0.00002",
    "--head-lr", "0.0002",
    "--weight-decay", "0.05",
    "--freeze-epochs", "2",
    "--resume"
]

print("Starting CAFR-Net v2 continuation...")
print("Starting checkpoint: Epoch 20")
print("Target: Epoch 30")
print("GPU:", __import__("torch").cuda.get_device_name(0))

subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
source = p.read_text(encoding="utf-8")

old = "model=CAFRNetV2(num_classes=num_classes,pretrained=True,**cfg)"
new = "model=CAFRNetV2(num_classes=num_classes,pretrained=False,**cfg)"

if old not in source:
    print("Expected model construction line not found.")
    print("Inspecting matching lines:")
    for line in source.splitlines():
        if "CAFRNetV2(" in line:
            print(repr(line))
else:
    source = source.replace(old, new, 1)
    p.write_text(source, encoding="utf-8")
    print("✅ Pretrained download disabled.")
    print("✅ Existing checkpoint will supply trained model weights.")

In [ ]:
import subprocess
import sys
import torch
from pathlib import Path

SOURCE = Path("/kaggle/working/CAFRNet_source")
WORK = Path("/kaggle/working/CAFRNet_continuation")

cmd = [
    sys.executable, "-u", str(SOURCE / "train_architecture.py"),
    "--arch", "cafrnet_v2",
    "--data", str(WORK / "cafr_data"),
    "--out", str(WORK / "cafrnet_seed42_continuation"),
    "--seed", "42",
    "--epochs", "30",
    "--batch-size", "16",
    "--workers", "2",
    "--image-size", "224",
    "--lambda-consistency", "0.25",
    "--encoder-lr", "0.00002",
    "--head-lr", "0.0002",
    "--weight-decay", "0.05",
    "--freeze-epochs", "2",
    "--resume"
]

print("GPU:", torch.cuda.get_device_name(0))
print("Resuming from checkpoint: Epoch 20")
print("Target: Epoch 30")

subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

TEST_ROOT = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/"
    "imagenette2-320/val"
)

print("Independent evaluation dataset:", TEST_ROOT)
print("Exists:", TEST_ROOT.exists())

if TEST_ROOT.exists():
    total = 0
    for folder in sorted(TEST_ROOT.iterdir()):
        if folder.is_dir():
            count = sum(
                1 for p in folder.iterdir()
                if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
            )
            total += count
            print(folder.name, ":", count, "images")

    print("\nTotal independent evaluation images:", total)

print("\nBEST CHECKPOINT:")
p = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "cafrnet_seed42_continuation/best_model.pt"
)
print("Exists:", p.exists())

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
lines = p.read_text(encoding="utf-8").splitlines()

for start, end in [(1, 135)]:
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

In [ ]:

import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from cafr_net import CAFRNetV2

# Paths
TEST_ROOT = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/"
    "imagenette2-320/val"
)

CHECKPOINT = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "cafrnet_seed42_continuation/best_model.pt"
)

RESULTS = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "independent_test_results"
)
RESULTS.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load checkpoint
ck = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

classes = ck["classes"]
config = ck["model_config"]

print("Checkpoint epoch:", ck["epoch"])
print("Checkpoint architecture:", ck["architecture"])
print("Model config:", config)

# Deterministic test preprocessing
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_ds = ImageFolder(
    str(TEST_ROOT),
    transform=test_transform
)

# Verify class ordering
assert test_ds.classes == classes, (
    f"Class mismatch!\n"
    f"Checkpoint: {classes}\n"
    f"Dataset: {test_ds.classes}"
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

# Reconstruct exact CAFR-Net v2 architecture
model = CAFRNetV2(
    num_classes=len(classes),
    pretrained=False,
    **config
)

model.load_state_dict(ck["model"], strict=True)
model = model.to(device)
model.eval()

# Independent inference
y_true = []
y_pred = []
all_probabilities = []

with torch.inference_mode():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.cpu().tolist())
        all_probabilities.extend(probabilities.cpu().numpy().tolist())

        if (batch_idx + 1) % 20 == 0:
            print(
                f"Processed {(batch_idx + 1) * 32}/"
                f"{len(test_ds)} images"
            )

# Metrics
accuracy = accuracy_score(y_true, y_pred)

macro_p, macro_r, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )
)

weighted_p, weighted_r, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_true, y_pred,
        average="weighted",
        zero_division=0
    )
)

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    output_dict=True,
    zero_division=0
)

cm = confusion_matrix(
    y_true, y_pred,
    labels=list(range(len(classes)))
)

print("\n" + "=" * 55)
print("CAFR-NET V2 — INDEPENDENT TEST RESULTS")
print("=" * 55)
print("Test images:", len(test_ds))
print("Correct predictions:", int(np.sum(np.array(y_true) == np.array(y_pred))))
print("Incorrect predictions:", int(np.sum(np.array(y_true) != np.array(y_pred))))
print(f"Accuracy:          {accuracy:.6f}")
print(f"Macro Precision:   {macro_p:.6f}")
print(f"Macro Recall:      {macro_r:.6f}")
print(f"Macro F1-score:    {macro_f1:.6f}")
print(f"Weighted Precision:{weighted_p:.6f}")
print(f"Weighted Recall:   {weighted_r:.6f}")
print(f"Weighted F1-score: {weighted_f1:.6f}")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,.0+
    labels=list(range(len(classes))),
    target_names=classes,
    zero_division=0
))

# Per-image predictions
predictions_df = pd.DataFrame({
    "image_path": [str(path) for path, _ in test_ds.samples],
    "true_class": [classes[i] for i in y_true],
    "predicted_class": [classes[i] for i in y_pred],
    "correct": [
        int(a == b) for a, b in zip(y_true, y_pred)
    ],
    "confidence": [
        float(max(prob)) for prob in all_probabilities
    ]
})

predictions_df.to_csv(
    RESULTS / "independent_test_predictions.csv",
    index=False
)

# Save confusion matrix
pd.DataFrame(
    cm,
    index=classes,
    columns=classes
).to_csv(RESULTS / "confusion_matrix.csv")

# Save per-class metrics
pd.DataFrame(report).transpose().to_csv(
    RESULTS / "classification_report.csv"
)

# Save summary
summary = {
    "experiment": "CAFR-Net v2 independent test evaluation",
    "checkpoint_epoch": int(ck["epoch"]),
    "seed": int(ck["seed"]),
    "architecture": ck["architecture"],
    "test_images": len(test_ds),
    "num_classes": len(classes),
    "accuracy": float(accuracy),
    "macro_precision": float(macro_p),
    "macro_recall": float(macro_r),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_p),
    "weighted_recall": float(weighted_r),
    "weighted_f1": float(weighted_f1),
    "class_names": classes
}

with open(RESULTS / "independent_test_metrics.json", "w") as f:
    json.dump(summary, f, indent=4)

print("\n✅ EVALUATION COMPLETED")
print("Results directory:", RESULTS)
print("Saved predictions, confusion matrix, classification report and metrics.")


In [ ]:
import sys
from pathlib import Path

SOURCE = Path("/kaggle/working/CAFRNet_source")
sys.path.insert(0, str(SOURCE))

from cafr_net import CAFRNetV2

print("✅ CAFRNetV2 imported successfully!")
print("Source:", SOURCE / "cafr_net.py")

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "independent_test_results"
)

df = pd.read_csv(
    RESULTS / "independent_test_predictions.csv"
)

errors = df[df["correct"] == 0].copy()

print("TOTAL MISCLASSIFICATIONS:", len(errors))
print("\nMISCLASSIFIED IMAGES:")
print(errors.to_string(index=False))

errors.to_csv(
    RESULTS / "misclassified_images.csv",
    index=False
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

RESULTS = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "independent_test_results"
)

df = pd.read_csv(RESULTS / "misclassified_images.csv")

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, row in df.iterrows():
    img = Image.open(row["image_path"]).convert("RGB")

    axes[i].imshow(img)
    axes[i].set_title(
        f"Actual: {row['true_class']}\n"
        f"Predicted: {row['predicted_class']}\n"
        f"Confidence: {row['confidence']:.4f}",
        fontsize=9
    )
    axes[i].axis("off")

plt.tight_layout()
plt.savefig(RESULTS / "misclassified_images_grid.png", dpi=180)
plt.show()

In [ ]:

import sys, json, io, zipfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter, ImageEnhance
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# --------------------------------------------------
# 1. PATHS AND MODEL
# --------------------------------------------------

SOURCE = Path("/kaggle/working/CAFRNet_source")
sys.path.insert(0, str(SOURCE))

from cafr_net import CAFRNetV2

TEST_ROOT = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/"
    "imagenette2-320/val"
)

CHECKPOINT = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "cafrnet_seed42_continuation/best_model.pt"
)

BASE_RESULTS = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "independent_test_results"
)

FINAL = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "FINAL_RESEARCH_ANALYSIS"
)
FINAL.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ck = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
classes = ck["classes"]

model = CAFRNetV2(
    num_classes=len(classes),
    pretrained=False,
    **ck["model_config"]
)
model.load_state_dict(ck["model"], strict=True)
model = model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

print("Model:", ck["architecture"])
print("Checkpoint epoch:", ck["epoch"])
print("Device:", device)
print("Classes:", len(classes))

# --------------------------------------------------
# 2. DATASET AND CORRUPTION DEFINITIONS
# --------------------------------------------------

class EvaluationDataset(Dataset):
    def __init__(self, root, corruption="clean"):
        self.base = ImageFolder(str(root))
        self.corruption = corruption
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        path, label = self.base.samples[index]
        image = self.base.loader(path).convert("RGB")

        if self.corruption == "noise_sigma20":
            arr = np.asarray(image).astype(np.float32)
            rng = np.random.default_rng(index + 420)
            noise = rng.normal(0, 20, arr.shape)
            image = Image.fromarray(
                np.uint8(np.clip(arr + noise, 0, 255))
            )

        elif self.corruption == "blur_radius2":
            image = image.filter(
                ImageFilter.GaussianBlur(radius=2.0)
            )

        elif self.corruption == "jpeg_quality20":
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=20)
            buffer.seek(0)
            image = Image.open(buffer).convert("RGB")

        elif self.corruption == "brightness_050":
            image = ImageEnhance.Brightness(image).enhance(0.50)

        elif self.corruption == "brightness_150":
            image = ImageEnhance.Brightness(image).enhance(1.50)

        return self.transform(image), label

# --------------------------------------------------
# 3. CALIBRATION METRICS
# --------------------------------------------------

def calibration_metrics(y_true, probabilities, bins=15):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    confidences = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    correctness = (predictions == y_true).astype(float)

    ece = 0.0
    edges = np.linspace(0, 1, bins + 1)

    for i in range(bins):
        mask = (
            (confidences >= edges[i]) &
            (confidences < edges[i + 1])
        )
        if i == bins - 1:
            mask |= confidences == 1.0

        if mask.any():
            bin_accuracy = correctness[mask].mean()
            bin_confidence = confidences[mask].mean()
            ece += mask.mean() * abs(
                bin_accuracy - bin_confidence
            )

    one_hot = np.eye(probabilities.shape[1])[y_true]
    brier = np.mean(
        np.sum((probabilities - one_hot) ** 2, axis=1)
    )

    selected = probabilities[np.arange(len(y_true)), y_true]
    nll = -np.log(np.clip(selected, 1e-12, 1)).mean()

    return {
        "ECE_15_bins": float(ece),
        "multiclass_Brier_score": float(brier),
        "negative_log_likelihood": float(nll)
    }

# --------------------------------------------------
# 4. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate(corruption):
    ds = EvaluationDataset(TEST_ROOT, corruption)
    loader = DataLoader(
        ds,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=(device.type == "cuda")
    )

    y_true, y_pred, probs_all = [], [], []

    with torch.inference_mode():
        for batch_index, (images, labels) in enumerate(loader):
            images = images.to(device, non_blocking=True)

            logits = model(images)
            probabilities = torch.softmax(logits, dim=1)
            predictions = logits.argmax(dim=1)

            y_true.extend(labels.numpy().tolist())
            y_pred.extend(predictions.cpu().tolist())
            probs_all.extend(probabilities.cpu().numpy().tolist())

            if (batch_index + 1) % 20 == 0:
                print(
                    corruption,
                    "processed:",
                    min((batch_index + 1) * 64, len(ds)),
                    "/",
                    len(ds)
                )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    probabilities = np.asarray(probs_all)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    result = {
        "condition": corruption,
        "images": len(ds),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        **calibration_metrics(y_true, probabilities)
    }

    return result, y_true, y_pred, probabilities

# --------------------------------------------------
# 5. CLEAN EVALUATION AND CONFUSION MATRIX
# --------------------------------------------------

conditions = [
    "clean",
    "noise_sigma20",
    "blur_radius2",
    "jpeg_quality20",
    "brightness_050",
    "brightness_150"
]

results = []
clean_data = None

for condition in conditions:
    print("\nEvaluating:", condition)
    result, yt, yp, probs = evaluate(condition)
    results.append(result)

    if condition == "clean":
        clean_data = (yt, yp, probs)

        cm = confusion_matrix(
            yt, yp, labels=list(range(len(classes)))
        )

        pd.DataFrame(
            cm, index=classes, columns=classes
        ).to_csv(FINAL / "confusion_matrix.csv")

        plt.figure(figsize=(12, 10))
        plt.imshow(cm, interpolation="nearest")
        plt.title("CAFR-Net v2 — Independent Test Confusion Matrix")
        plt.colorbar()
        plt.xticks(range(len(classes)), classes, rotation=90)
        plt.yticks(range(len(classes)), classes)

        for i in range(len(classes)):
            for j in range(len(classes)):
                plt.text(j, i, str(cm[i, j]),
                         ha="center", va="center", fontsize=8)

        plt.xlabel("Predicted class")
        plt.ylabel("Actual class")
        plt.tight_layout()
        plt.savefig(
            FINAL / "confusion_matrix.png",
            dpi=200
        )
        plt.close()

        report = classification_report(
            yt, yp,
            labels=list(range(len(classes))),
            target_names=classes,
            output_dict=True,
            zero_division=0
        )

        pd.DataFrame(report).transpose().to_csv(
            FINAL / "per_class_classification_report.csv"
        )

        # Save clean predictions and confidence
        clean_predictions = pd.DataFrame({
            "true_class": [classes[x] for x in yt],
            "predicted_class": [classes[x] for x in yp],
            "correct": (yt == yp).astype(int),
            "confidence": probs.max(axis=1)
        })
        clean_predictions.to_csv(
            FINAL / "clean_test_predictions.csv",
            index=False
        )

# --------------------------------------------------
# 6. ROBUSTNESS RESULTS
# --------------------------------------------------

robustness_df = pd.DataFrame(results)
robustness_df.to_csv(
    FINAL / "corruption_robustness_results.csv",
    index=False
)

print("\nROBUSTNESS RESULTS")
print(robustness_df.to_string(index=False))

# Accuracy comparison chart
plt.figure(figsize=(12, 6))
plt.bar(
    robustness_df["condition"],
    robustness_df["accuracy"] * 100
)
plt.ylabel("Accuracy (%)")
plt.xlabel("Evaluation condition")
plt.title("CAFR-Net v2 — Corruption Stress Evaluation")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FINAL / "robustness_accuracy.png", dpi=200)
plt.close()

# --------------------------------------------------
# 7. CALIBRATION SUMMARY
# --------------------------------------------------

calibration_df = robustness_df[
    [
        "condition",
        "ECE_15_bins",
        "multiclass_Brier_score",
        "negative_log_likelihood"
    ]
]
calibration_df.to_csv(
    FINAL / "calibration_results.csv",
    index=False
)

# --------------------------------------------------
# 8. MISCLASSIFICATION IMAGE GRID
# --------------------------------------------------

if clean_data is not None:
    yt, yp, probs = clean_data

    base_predictions = pd.read_csv(
        BASE_RESULTS / "independent_test_predictions.csv"
    )
    errors = base_predictions[
        base_predictions["correct"] == 0
    ].copy()

    errors.to_csv(
        FINAL / "misclassified_images.csv",
        index=False
    )

    if len(errors):
        fig, axes = plt.subplots(
            2, 4, figsize=(18, 10)
        )
        axes = axes.flatten()

        for i, (_, row) in enumerate(errors.iterrows()):
            image = Image.open(row["image_path"]).convert("RGB")
            axes[i].imshow(image)
            axes[i].set_title(
                f"True: {row['true_class']}\n"
                f"Pred: {row['predicted_class']}\n"
                f"Confidence: {row['confidence']:.4f}",
                fontsize=9
            )
            axes[i].axis("off")

        for i in range(len(errors), len(axes)):
            axes[i].axis("off")

        plt.tight_layout()
        plt.savefig(
            FINAL / "misclassified_images_grid.png",
            dpi=200
        )
        plt.close()

# --------------------------------------------------
# 9. RESEARCH SUMMARY
# --------------------------------------------------

summary = {
    "experiment": "CAFR-Net v2 — Consolidated Evaluation",
    "architecture": ck["architecture"],
    "checkpoint_epoch": int(ck["epoch"]),
    "seed": int(ck["seed"]),
    "dataset": "Imagenette2-320 official val directory",
    "test_images": 3925,
    "classes": classes,
    "model_config": ck["model_config"],
    "conditions": results,
    "notes": [
        "Clean evaluation uses deterministic resize and ImageNet normalization.",
        "Corruption tests use stronger settings than the training augmentation ranges.",
        "Corruption tests are stress evaluations, not a separate external dataset.",
        "Ablation and baseline comparisons require separately trained checkpoints."
    ]
}

with open(FINAL / "final_research_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

# --------------------------------------------------
# 10. BACKUP RESULTS AND SOURCE
# --------------------------------------------------

shutil.copy2(
    SOURCE / "cafr_net.py",
    FINAL / "cafr_net.py"
)
shutil.copy2(
    SOURCE / "train_architecture.py",
    FINAL / "train_architecture.py"
)
shutil.copy2(
    CHECKPOINT,
    FINAL / "best_model_epoch30.pt"
)

zip_path = Path(
    "/kaggle/working/CAFRNet_FINAL_RESEARCH_BACKUP.zip"
)

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for file in FINAL.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(FINAL))

print("\n" + "=" * 60)
print("ALL CONSOLIDATED EVALUATIONS COMPLETED")
print("=" * 60)
print("Results folder:", FINAL)
print("Backup ZIP:", zip_path)
print("Backup size MB:", round(zip_path.stat().st_size / 1024**2, 2))
print("\nFinal robustness table:")
print(robustness_df.to_string(index=False))


In [ ]:
from pathlib import Path
import zipfile

WORK = Path("/kaggle/working/CAFRNet_continuation")
FINAL = WORK / "FINAL_RESEARCH_ANALYSIS"
OUT = WORK / "cafrnet_seed42_continuation"

zip_path = Path("/kaggle/working/CAFRNet_FINAL_RESEARCH_BACKUP.zip")

extra_files = [
    OUT / "latest_model.pt",
    OUT / "best_model.pt",
    Path("/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42/latest_model.pt"),
    Path("/kaggle/input/datasets/maddulaadithya/seed42/cafrnet_seed42/best_model.pt"),
    FINAL / "final_research_summary.json",
]

with zipfile.ZipFile(zip_path, "a", zipfile.ZIP_DEFLATED) as z:
    for p in extra_files:
        if p.exists():
            z.write(p, arcname="checkpoints/" + p.name)
            print("Added:", p.name)
        else:
            print("Missing:", p)

print("\nFinal ZIP size (MB):", round(zip_path.stat().st_size / 1024**2, 2))

In [ ]:
import zipfile
from collections import Counter
from pathlib import Path

ZIP_PATH = Path("/kaggle/working/CAFRNet_FINAL_RESEARCH_BACKUP.zip")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()
    counts = Counter(names)

    print("ZIP integrity:", z.testzip() or "PASSED")
    print("Total entries:", len(names))

    print("\nDuplicate entries:")
    duplicates = {name: count for name, count in counts.items() if count > 1}
    print(duplicates if duplicates else "None")

    print("\nCheckpoint files:")
    for name in names:
        if "checkpoint" in name.lower():
            info = z.getinfo(name)
            print(name, f"{info.file_size / (1024**2):.2f} MB")

In [ ]:
import zipfile
from pathlib import Path

source = Path("/kaggle/working/CAFRNet_FINAL_RESEARCH_BACKUP.zip")
clean = Path("/kaggle/working/CAFRNet_FINAL_RESEARCH_BACKUP_CLEAN.zip")

with zipfile.ZipFile(source, "r") as zin:
    seen = set()

    with zipfile.ZipFile(clean, "w", zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            if item.filename in seen:
                continue

            seen.add(item.filename)
            zout.writestr(item.filename, zin.read(item))

with zipfile.ZipFile(clean, "r") as z:
    print("Integrity:", z.testzip() or "PASSED")
    print("Total entries:", len(z.namelist()))
    print("Final size (MB):", round(clean.stat().st_size / (1024**2), 2))
    print("\nFiles:")
    for name in z.namelist():
        print(name)

In [ ]:
from pathlib import Path
import torch
import re

SRC = Path("/kaggle/working/CAFRNet_source")
OUT = Path("/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation")

train_file = SRC / "train_architecture.py"
model_file = SRC / "cafr_net.py"
checkpoint = OUT / "best_model.pt"

print("=== ENVIRONMENT ===")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("PyTorch:", torch.__version__)

print("\n=== FILES ===")
for p in [train_file, model_file, checkpoint]:
    print(p, "EXISTS" if p.exists() else "MISSING")

if not all(p.exists() for p in [train_file, model_file, checkpoint]):
    raise FileNotFoundError("A required source/checkpoint file is missing.")

code = train_file.read_text()

print("\n=== ABLATION OPTIONS ===")
for line in code.splitlines():
    if "--ablation" in line or "no_edge" in line or "no_semantic_fusion" in line or "no_attention" in line:
        print(line.strip())

print("\n=== CHECKPOINT ===")
ck = torch.load(checkpoint, map_location="cpu", weights_only=False)
print("Epoch:", ck.get("epoch"))
print("Architecture:", ck.get("architecture"))
print("Seed:", ck.get("seed"))
print("Model config:", ck.get("model_config"))

print("\n=== MODEL INITIALIZATION / ABLATION CODE ===")
for i, line in enumerate(code.splitlines()):
    if any(k in line for k in ["def build_model", "def make_model", "ablation", "load_state_dict", "pretrained"]):
        start, end = max(0, i-2), min(len(code.splitlines()), i+5)
        print(f"\n--- lines {start+1}-{end} ---")
        print("\n".join(f"{j+1}: {code.splitlines()[j]}" for j in range(start, end)))

print("\n=== CLI OPTIONS ===")
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(train_file), "--help"],
    capture_output=True, text=True
)
print(result.stdout[:12000])
if result.returncode != 0:
    print("HELP ERROR:", result.stderr[-3000:])

In [ ]:
import sys, torch
from pathlib import Path

SRC = Path("/kaggle/working/CAFRNet_source")
sys.path.insert(0, str(SRC))

from cafr_net import CAFRNetV2

CKPT = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "cafrnet_seed42_continuation/best_model.pt"
)

ck = torch.load(CKPT, map_location="cpu", weights_only=False)
full_state = ck["model"]

configs = {
    "full": {},
    "no_edge": {"use_edge": False},
    "no_semantic_fusion": {"use_semantic_fusion": False},
    "no_attention": {"use_attention": False},
}

print("Checkpoint epoch:", ck.get("epoch"))
print("Full checkpoint tensors:", len(full_state))

for name, cfg in configs.items():
    model = CAFRNetV2(
        num_classes=len(ck["class_names"]),
        pretrained=False,
        **{
            "use_edge": True,
            "use_semantic_fusion": True,
            "use_attention": True,
            **cfg
        }
    )
    target = model.state_dict()

    compatible = [
        k for k, v in full_state.items()
        if k in target and target[k].shape == v.shape
    ]
    missing = [k for k in target if k not in compatible]
    mismatched = [
        k for k, v in full_state.items()
        if k in target and target[k].shape != v.shape
    ]

    encoder_keys = [k for k in compatible if k.startswith("encoder.")]
    print(f"\n{name}")
    print("Target tensors:", len(target))
    print("Compatible tensors:", len(compatible))
    print("Compatible encoder tensors:", len(encoder_keys))
    print("Missing tensors:", len(missing))
    print("Shape mismatches:", len(mismatched))
    print("Sample missing:", missing[:8])

    del model

print("\nCompatibility check completed.")

In [ ]:
import sys, torch
from pathlib import Path

SRC = Path("/kaggle/working/CAFRNet_source")
sys.path.insert(0, str(SRC))

from cafr_net import CAFRNetV2

CKPT = Path(
    "/kaggle/working/CAFRNet_continuation/"
    "cafrnet_seed42_continuation/best_model.pt"
)

DATA = Path("/kaggle/working/CAFRNet_continuation/cafr_data")

ck = torch.load(CKPT, map_location="cpu", weights_only=False)
full_state = ck["model"]

num_classes = len([p for p in (DATA / "train").iterdir() if p.is_dir()])

print("Checkpoint epoch:", ck.get("epoch"))
print("Number of classes:", num_classes)
print("Full checkpoint tensors:", len(full_state))

configs = {
    "full": {},
    "no_edge": {"use_edge": False},
    "no_semantic_fusion": {"use_semantic_fusion": False},
    "no_attention": {"use_attention": False},
}

for name, cfg in configs.items():
    model = CAFRNetV2(
        num_classes=num_classes,
        pretrained=False,
        **{
            "use_edge": True,
            "use_semantic_fusion": True,
            "use_attention": True,
            **cfg
        }
    )

    target = model.state_dict()

    compatible = [
        k for k, v in full_state.items()
        if k in target and target[k].shape == v.shape
    ]

    missing = [k for k in target if k not in compatible]

    mismatched = [
        k for k, v in full_state.items()
        if k in target and target[k].shape != v.shape
    ]

    encoder_keys = [
        k for k in compatible if k.startswith("encoder.")
    ]

    print(f"\n{name}")
    print("Target tensors:", len(target))
    print("Compatible tensors:", len(compatible))
    print("Compatible encoder tensors:", len(encoder_keys))
    print("Missing tensors:", len(missing))
    print("Shape mismatches:", len(mismatched))
    print("Sample missing:", missing[:8])

    del model

print("\nCompatibility check completed.")

In [ ]:
from pathlib import Path

SRC = Path("/kaggle/working/CAFRNet_source")

for filename in ["cafr_net.py", "train_architecture.py"]:
    path = SRC / filename
    lines = path.read_text().splitlines()

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    keywords = (
        "use_edge",
        "use_semantic_fusion",
        "use_attention",
        "def forward",
        "load_state_dict",
        "optimizer.load_state_dict",
        "start_epoch",
    )

    shown = set()

    for i, line in enumerate(lines):
        if any(k in line for k in keywords):
            start = max(0, i - 3)
            end = min(len(lines), i + 7)

            if any(j in shown for j in range(start, end)):
                continue

            shown.update(range(start, end))
            print(f"\n--- Lines {start+1}-{end} ---")
            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
s = p.read_text()

if "--init-weights" not in s:
    marker = 'ap.add_argument("--resume",action="store_true",help="Resume from OUT/latest_model.pt")'
    if marker not in s:
        raise RuntimeError("Could not locate resume argument. No changes made.")

    s = s.replace(
        marker,
        marker + '\nap.add_argument("--init-weights", type=str, default=None, help="Initialize model weights without restoring optimizer state.")'
    )

marker = "    model=model.to(device)\n"
if "Warm-started model weights from" not in s:
    if marker not in s:
        raise RuntimeError("Could not locate model initialization point. No changes made.")

    addition = '''
    model=model.to(device)
    if args.init_weights:
        init_ck=torch.load(args.init_weights,map_location="cpu",weights_only=False)
        init_state=init_ck["model"]
        current_state=model.state_dict()
        compatible={
            k:v for k,v in init_state.items()
            if k in current_state and current_state[k].shape==v.shape
        }
        if len(compatible)!=len(current_state):
            raise RuntimeError(
                f"Unsafe initialization: {len(compatible)}/{len(current_state)} tensors compatible."
            )
        model.load_state_dict(compatible,strict=True)
        print(
            f"Warm-started model weights from {args.init_weights}; "
            f"loaded {len(compatible)} tensors. Optimizer starts fresh.",
            flush=True
        )
'''
    s = s.replace(marker, addition, 1)

p.write_text(s)
print("Updated training script:", p)

In [ ]:
import subprocess, sys

p = "/kaggle/working/CAFRNet_source/train_architecture.py"

r = subprocess.run(
    [sys.executable, p, "--help"],
    capture_output=True, text=True
)

print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("Training script validation failed.")

assert "--init-weights" in r.stdout
print("TRAINING SCRIPT VALIDATED")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")

# Preserve the current version before repairing
backup = p.with_name("train_architecture_damaged_backup.py")
backup.write_text(p.read_text())

lines = p.read_text().splitlines()

print("Backup:", backup)
print("\nTRAINING SCRIPT LINES 145–175:\n")

for i in range(144, min(175, len(lines))):
    print(f"{i+1}: {lines[i]!r}")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")

s = p.read_text()

bad = 'ap.add_argument("--init-weights", type=str, default=None, help="Initialize model weights without restoring optimizer state.")'
good = '    ' + bad

assert s.count(bad) == 1, "Expected exactly one damaged line."

p.write_text(s.replace(bad, good))

print("Indentation repaired.")

In [ ]:
import subprocess, sys

p = "/kaggle/working/CAFRNet_source/train_architecture.py"

r = subprocess.run(
    [sys.executable, p, "--help"],
    capture_output=True,
    text=True
)

print(r.stdout)

if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("Validation failed.")

assert "--init-weights" in r.stdout
print("SUCCESS: Training script is valid.")

In [ ]:
import subprocess
import sys
from pathlib import Path

SCRIPT = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
DATA = Path("/kaggle/working/CAFRNet_continuation/cafr_data")
INIT = Path("/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt")

BASE = Path("/kaggle/working/CAFRNet_ABLATIONS")
BASE.mkdir(parents=True, exist_ok=True)

experiments = [
    "no_edge",
    "no_semantic_fusion",
    "no_attention"
]

for ablation in experiments:
    out = BASE / ablation
    out.mkdir(parents=True, exist_ok=True)

    latest = out / "latest_model.pt"

    cmd = [
        sys.executable, "-u", str(SCRIPT),
        "--arch", "cafrnet_v2",
        "--ablation", ablation,
        "--data", str(DATA),
        "--out", str(out),
        "--seed", "42",
        "--epochs", "10",
        "--batch-size", "16",
        "--workers", "2",
        "--image-size", "224",
        "--lambda-consistency", "0.25",
        "--encoder-lr", "0.00002",
        "--head-lr", "0.0002",
        "--weight-decay", "0.05",
        "--freeze-epochs", "0"
    ]

    if latest.exists():
        print(f"\nResuming {ablation} from its own checkpoint.")
        cmd.append("--resume")
    else:
        print(f"\nInitializing {ablation} from full-model epoch 30.")
        cmd.extend(["--init-weights", str(INIT)])

    print("\n" + "=" * 65)
    print("EXPERIMENT:", ablation)
    print("=" * 65)

    result = subprocess.run(cmd)

    if result.returncode != 0:
        print(f"\nSTOPPED: {ablation}")
        print("Return code:", result.returncode)
        print("Its checkpoint, if created, is retained for resumption.")
        break

    print(f"\nCOMPLETED: {ablation}")

print("\nABLATION RUNNER FINISHED.")

In [ ]:
from pathlib import Path

cache = Path.home() / ".cache/torch/hub/checkpoints"

print("=== PRETRAINED WEIGHT CACHE ===")
if cache.exists():
    for f in cache.glob("*"):
        print(f.name, round(f.stat().st_size / (1024**2), 2), "MB")
else:
    print("No pretrained-weight cache found.")

print("\n=== BASELINE MODEL DEFINITIONS ===")
p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
lines = p.read_text().splitlines()

for i in range(95, min(125, len(lines))):
    print(f"{i+1}: {lines[i]}")

In [ ]:
import torch
from torchvision.models import (
    ResNet18_Weights,
    ConvNeXt_Tiny_Weights
)
from pathlib import Path

cache = Path.home() / ".cache/torch/hub/checkpoints"
cache.mkdir(parents=True, exist_ok=True)

models = {
    "ResNet18": ResNet18_Weights.DEFAULT,
    "ConvNeXt_Tiny": ConvNeXt_Tiny_Weights.DEFAULT
}

for name, weights in models.items():
    print(f"\nChecking {name} pretrained weights...")

    try:
        state = weights.get_state_dict(
            progress=True,
            check_hash=True
        )
        print(f"SUCCESS: {name}")
        print("Parameters loaded:", len(state))
        del state

    except Exception as e:
        print(f"DOWNLOAD FAILED: {name}")
        print(type(e).__name__, str(e)[:500])
        break

print("\nCached files:")
for f in cache.glob("*"):
    print(f.name, round(f.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/input"),
    Path("/root/.cache"),
    Path("/kaggle/working")
]

terms = ["resnet18", "resnet-18", "convnext_tiny", "convnext-tiny"]

for root in roots:
    print("\nSEARCHING:", root)
    if not root.exists():
        continue

    found = 0
    for p in root.rglob("*"):
        if p.is_file() and any(t in p.name.lower() for t in terms):
            print(p, round(p.stat().st_size / 1024**2, 2), "MB")
            found += 1
            if found >= 20:
                break

    if found == 0:
        print("No matching weight files found.")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/CAFRNet_source/train_architecture.py")
s = p.read_text()

old_conv = '        model=convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)'
new_conv = '''        model=convnext_tiny(weights=None)
        pretrained_state=torch.load(
            "/kaggle/input/datasets/maddulaadithya/imagenet-pretrained-baseline-weights/convnext_tiny-983f1562.pth",
            map_location="cpu", weights_only=True
        )
        model.load_state_dict(pretrained_state, strict=True)'''

old_res = '        model=resnet18(weights=ResNet18_Weights.DEFAULT)'
new_res = '''        model=resnet18(weights=None)
        pretrained_state=torch.load(
            "/kaggle/input/datasets/maddulaadithya/imagenet-pretrained-baseline-weights/resnet18-f37072fd.pth",
            map_location="cpu", weights_only=True
        )
        model.load_state_dict(pretrained_state, strict=True)'''

assert s.count(old_conv) == 1, "ConvNeXt line not found exactly once."
assert s.count(old_res) == 1, "ResNet line not found exactly once."

s = s.replace(old_conv, new_conv)
s = s.replace(old_res, new_res)
p.write_text(s)

print("Baseline loading patched successfully.")

In [ ]:
import torch
from torchvision.models import convnext_tiny, resnet18

weights_dir = "/kaggle/input/datasets/maddulaadithya/imagenet-pretrained-baseline-weights"

checks = [
    (
        "ResNet18",
        resnet18(weights=None),
        f"{weights_dir}/resnet18-f37072fd.pth"
    ),
    (
        "ConvNeXt-Tiny",
        convnext_tiny(weights=None),
        f"{weights_dir}/convnext_tiny-983f1562.pth"
    )
]

for name, model, path in checks:
    state = torch.load(path, map_location="cpu", weights_only=True)
    result = model.load_state_dict(state, strict=True)
    print(name, "PASS —", result)
    del model, state

print("\nBoth pretrained architectures validated.")

In [ ]:
import subprocess, sys

script = "/kaggle/working/CAFRNet_source/train_architecture.py"

r = subprocess.run(
    [sys.executable, script, "--help"],
    capture_output=True, text=True
)

if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("Training script validation failed.")

print("Training script validated.")

In [ ]:
import subprocess, sys
from pathlib import Path

SCRIPT = "/kaggle/working/CAFRNet_source/train_architecture.py"
DATA = "/kaggle/working/CAFRNet_continuation/cafr_data"

BASE = Path("/kaggle/working/CAFRNet_BASELINES")
BASE.mkdir(parents=True, exist_ok=True)

for arch in ["resnet18", "convnext_tiny"]:
    out = BASE / arch
    out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "-u", SCRIPT,
        "--arch", arch,
        "--data", DATA,
        "--out", str(out),
        "--seed", "42",
        "--epochs", "30",
        "--batch-size", "16",
        "--workers", "2",
        "--image-size", "224",
        "--lambda-consistency", "0.25",
        "--encoder-lr", "0.00002",
        "--head-lr", "0.0002",
        "--weight-decay", "0.05",
        "--freeze-epochs", "2"
    ]

    if (out / "latest_model.pt").exists():
        cmd.append("--resume")
        print(f"\nResuming {arch} from its checkpoint.")
    else:
        print(f"\nStarting pretrained {arch} baseline.")

    print("=" * 60)
    result = subprocess.run(cmd)

    if result.returncode != 0:
        print(f"STOPPED: {arch}, return code {result.returncode}")
        break

    print(f"COMPLETED: {arch}")

print("\nBASELINE TRAINING LOOP FINISHED.")

In [ ]:
from pathlib import Path
import torch

paths = {
    "CAFR-Net Full": "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
    "No Edge": "/kaggle/working/CAFRNet_ABLATIONS/no_edge/best_model.pt",
    "No Semantic Fusion": "/kaggle/working/CAFRNet_ABLATIONS/no_semantic_fusion/best_model.pt",
    "No Attention": "/kaggle/working/CAFRNet_ABLATIONS/no_attention/best_model.pt",
    "ResNet-18": "/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt",
    "ConvNeXt-Tiny": "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
}

for name, path in paths.items():
    p = Path(path)
    if not p.exists():
        print(f"MISSING: {name} — {path}")
        continue
    ck = torch.load(p, map_location="cpu", weights_only=False)
    print(f"OK: {name} | {p.stat().st_size / (1024**2):.2f} MB | keys: {list(ck.keys())[:8]}")

In [ ]:
from pathlib import Path
import torch
import inspect
import sys

CHECKPOINTS = {
    "CAFR-Net Full": "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
    "No Edge": "/kaggle/working/CAFRNet_ABLATIONS/no_edge/best_model.pt",
    "No Semantic Fusion": "/kaggle/working/CAFRNet_ABLATIONS/no_semantic_fusion/best_model.pt",
    "No Attention": "/kaggle/working/CAFRNet_ABLATIONS/no_attention/best_model.pt",
    "ResNet-18": "/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt",
    "ConvNeXt-Tiny": "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
}

print("=" * 65)
print("CHECKPOINT CONFIGURATION INSPECTION")
print("=" * 65)

for name, path in CHECKPOINTS.items():
    ck = torch.load(path, map_location="cpu", weights_only=False)

    print(f"\n{name}")
    print("-" * 50)
    print("Architecture:", ck.get("architecture"))
    print("Epoch:", ck.get("epoch"))
    print("Validation accuracy:", ck.get("val_accuracy"))
    print("Classes:", ck.get("classes"))
    print("Model config:", ck.get("model_config"))

print("\n" + "=" * 65)
print("SOURCE FILES")
print("=" * 65)

sys.path.insert(0, "/kaggle/working/CAFRNet_source")

for file in [
    "/kaggle/working/CAFRNet_source/cafr_net.py",
    "/kaggle/working/CAFRNet_source/train_architecture.py"
]:
    p = Path(file)
    print(f"{p.name}: {'FOUND' if p.exists() else 'MISSING'}")

print("\nOfficial evaluation directory:")
val_dir = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/"
    "imagenette2-320/val"
)
print("Exists:", val_dir.exists())
if val_dir.exists():
    print("Classes:", len([x for x in val_dir.iterdir() if x.is_dir()]))
    print("Class folders:", sorted(x.name for x in val_dir.iterdir() if x.is_dir()))

print("\nINSPECTION COMPLETE")

In [ ]:
import ast
from pathlib import Path

files = [
    Path("/kaggle/working/CAFRNet_source/cafr_net.py"),
    Path("/kaggle/working/CAFRNet_source/train_architecture.py")
]

keywords = (
    "class ", "def ", "transform", "Normalize",
    "Resize", "CenterCrop", "build_model",
    "resnet18", "convnext_tiny", "CAFRNet"
)

for path in files:
    print("\n" + "=" * 80)
    print("FILE:", path.name)
    print("=" * 80)

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            print(f"\n{type(node).__name__}: {node.name} (line {node.lineno})")
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                print("Signature:", ast.get_source_segment(source, node.args))

    print("\n--- Relevant source lines ---")
    shown = set()
    for i, line in enumerate(lines):
        if any(k.lower() in line.lower() for k in keywords):
            start = max(0, i - 2)
            end = min(len(lines), i + 5)
            for j in range(start, end):
                if j not in shown:
                    print(f"{j+1:4}: {lines[j]}")
                    shown.add(j)
            print()

In [ ]:

import os, sys, io, gc, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

# ---------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ---------------------------------------------------------
SOURCE = "/kaggle/working/CAFRNet_source"
sys.path.insert(0, SOURCE)
from train_architecture import build_model

VAL_DIR = Path(
    "/kaggle/input/datasets/xbinchen/imagenette2-320/"
    "imagenette2-320/val"
)

OUT = Path("/kaggle/working/CAFRNet_FINAL_MODEL_COMPARISON")
OUT.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = {
    "CAFRNet_Full": (
        "/kaggle/working/CAFRNet_continuation/"
        "cafrnet_seed42_continuation/best_model.pt",
        "cafrnet_v2", "full"
    ),
    "CAFRNet_No_Edge": (
        "/kaggle/working/CAFRNet_ABLATIONS/no_edge/best_model.pt",
        "cafrnet_v2", "no_edge"
    ),
    "CAFRNet_No_Semantic_Fusion": (
        "/kaggle/working/CAFRNet_ABLATIONS/no_semantic_fusion/best_model.pt",
        "cafrnet_v2", "no_semantic_fusion"
    ),
    "CAFRNet_No_Attention": (
        "/kaggle/working/CAFRNet_ABLATIONS/no_attention/best_model.pt",
        "cafrnet_v2", "no_attention"
    ),
    "ResNet18": (
        "/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt",
        "resnet18", "full"
    ),
    "ConvNeXt_Tiny": (
        "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
        "convnext_tiny", "full"
    )
}

CONDITIONS = [
    "clean", "noise_sigma20", "blur_radius2",
    "jpeg_quality20", "brightness_050", "brightness_150"
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64 if DEVICE.type == "cuda" else 16
NUM_WORKERS = 2

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

print("Device:", DEVICE)
print("Evaluation images:", sum(
    1 for p in VAL_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in
    {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
))
print("Models:", len(CHECKPOINTS))
print("Conditions:", len(CONDITIONS))

# ---------------------------------------------------------
# 2. DETERMINISTIC EVALUATION DATASET
# ---------------------------------------------------------
class RobustnessDataset(Dataset):
    def __init__(self, root, condition):
        from torchvision.datasets import ImageFolder
        self.base = ImageFolder(str(root))
        self.samples = self.base.samples
        self.classes = self.base.classes
        self.condition = condition
        self.resize = transforms.Resize((224, 224))
        self.tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD)
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        image = self.resize(image)
        condition = self.condition

        if condition == "noise_sigma20":
            arr = np.asarray(image).astype(np.float32)
            rng = np.random.default_rng(42 + index)
            noise = rng.normal(0, 20, arr.shape)
            arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
            image = Image.fromarray(arr)

        elif condition == "blur_radius2":
            image = image.filter(ImageFilter.GaussianBlur(radius=2))

        elif condition == "jpeg_quality20":
            buffer = io.BytesIO()
            image.save(buffer, format="JPEG", quality=20)
            buffer.seek(0)
            image = Image.open(buffer).convert("RGB")

        elif condition == "brightness_050":
            image = ImageEnhance.Brightness(image).enhance(0.5)

        elif condition == "brightness_150":
            image = ImageEnhance.Brightness(image).enhance(1.5)

        return self.tensor(image), label, path

# ---------------------------------------------------------
# 3. METRIC HELPERS
# ---------------------------------------------------------
def calculate_ece(probs, targets, bins=15):
    confidence = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == targets).astype(np.float64)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0

    for i in range(bins):
        mask = (confidence > edges[i]) & (confidence <= edges[i + 1])
        if mask.any():
            ece += mask.mean() * abs(
                correct[mask].mean() - confidence[mask].mean()
            )
    return float(ece)

# ---------------------------------------------------------
# 4. RUN ALL MODELS ON ALL CONDITIONS
# ---------------------------------------------------------
summary_rows = []
class_rows = []
prediction_rows = []
n_classes = 10

for model_name, (checkpoint_path, arch, ablation) in CHECKPOINTS.items():

    print("\n" + "=" * 75)
    print("LOADING MODEL:", model_name)
    print("=" * 75)

    ck = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )

    classes = ck["classes"]
    if len(classes) != n_classes:
        raise ValueError(f"Unexpected class count in {model_name}")

    model, _, _, _ = build_model(
        arch, num_classes=n_classes, ablation=ablation
    )

    model.load_state_dict(ck["model"], strict=True)
    model = model.to(DEVICE)
    model.eval()

    for condition in CONDITIONS:
        start = time.time()
        dataset = RobustnessDataset(VAL_DIR, condition)

        if dataset.classes != classes:
            raise ValueError(
                f"Class order mismatch for {model_name}: "
                f"{dataset.classes} vs {classes}"
            )

        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda")
        )

        all_targets = []
        all_predictions = []
        all_probabilities = []
        all_paths = []

        with torch.inference_mode():
            for images, labels, paths in loader:
                images = images.to(DEVICE, non_blocking=True)
                logits = model(images)

                probabilities = torch.softmax(logits.float(), dim=1)
                predictions = probabilities.argmax(dim=1)

                all_targets.extend(labels.numpy().tolist())
                all_predictions.extend(predictions.cpu().numpy().tolist())
                all_probabilities.append(probabilities.cpu().numpy())
                all_paths.extend(paths)

        y_true = np.asarray(all_targets, dtype=np.int64)
        y_pred = np.asarray(all_predictions, dtype=np.int64)
        probs = np.concatenate(all_probabilities, axis=0)

        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )
        wp, wr, wf1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="weighted", zero_division=0
        )

        one_hot = np.eye(n_classes)[y_true]
        brier = float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))
        nll = float(-np.log(
            np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, 1)
        ).mean())
        ece = calculate_ece(probs, y_true)

        summary_rows.append({
            "model": model_name,
            "condition": condition,
            "images": len(y_true),
            "accuracy": accuracy,
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1,
            "weighted_precision": wp,
            "weighted_recall": wr,
            "weighted_f1": wf1,
            "ece_15_bins": ece,
            "brier_score": brier,
            "nll": nll
        })

        report = classification_report(
            y_true, y_pred,
            labels=list(range(n_classes)),
            target_names=classes,
            output_dict=True,
            zero_division=0
        )
        for class_name in classes:
            row = report[class_name]
            class_rows.append({
                "model": model_name,
                "condition": condition,
                "class": class_name,
                "precision": row["precision"],
                "recall": row["recall"],
                "f1_score": row["f1-score"],
                "support": row["support"]
            })

        cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
        pd.DataFrame(cm, index=classes, columns=classes).to_csv(
            OUT / f"{model_name}_{condition}_confusion_matrix.csv"
        )

        for i in range(len(y_true)):
            prediction_rows.append({
                "model": model_name,
                "condition": condition,
                "image": all_paths[i],
                "true_class": classes[y_true[i]],
                "predicted_class": classes[y_pred[i]],
                "correct": bool(y_true[i] == y_pred[i]),
                "confidence": float(probs[i].max())
            })

        elapsed = time.time() - start
        print(
            f"{model_name:27s} | {condition:20s} | "
            f"Accuracy: {accuracy*100:.4f}% | "
            f"Macro F1: {f1:.5f} | {elapsed:.1f}s"
        )

        del dataset, loader, probs, all_probabilities
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del model, ck
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ---------------------------------------------------------
# 5. SAVE FINAL COMPARISON RESULTS
# ---------------------------------------------------------
summary = pd.DataFrame(summary_rows)
class_report = pd.DataFrame(class_rows)
predictions = pd.DataFrame(prediction_rows)

summary.to_csv(OUT / "all_models_robustness_comparison.csv", index=False)
class_report.to_csv(OUT / "all_models_per_class_metrics.csv", index=False)
predictions.to_csv(OUT / "all_models_predictions.csv", index=False)

clean = summary[summary["condition"] == "clean"].copy()
clean = clean.sort_values("accuracy", ascending=False)
clean.to_csv(OUT / "clean_model_comparison.csv", index=False)

robustness = summary.pivot(
    index="model", columns="condition", values="accuracy"
)
robustness.to_csv(OUT / "robustness_accuracy_matrix.csv")

metadata = {
    "evaluation_dataset": str(VAL_DIR),
    "images_per_condition": int(len(RobustnessDataset(VAL_DIR, "clean"))),
    "models": list(CHECKPOINTS.keys()),
    "conditions": CONDITIONS,
    "device": str(DEVICE),
    "image_size": 224,
    "normalization_mean": MEAN,
    "normalization_std": STD,
    "note": (
        "All models evaluated on the same official validation images. "
        "Corruptions are controlled stress tests, not external datasets."
    )
}
with open(OUT / "evaluation_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 75)
print("FINAL CLEAN MODEL COMPARISON")
print("=" * 75)
print(clean[
    ["model", "accuracy", "macro_precision", "macro_recall", "macro_f1"]
].to_string(index=False))

print("\n" + "=" * 75)
print("ROBUSTNESS ACCURACY MATRIX (%)")
print("=" * 75)
print((robustness * 100).round(4).to_string())

print("\nALL EVALUATIONS COMPLETED.")
print("Results folder:", OUT)
print("Summary:", OUT / "all_models_robustness_comparison.csv")
print("Clean comparison:", OUT / "clean_model_comparison.csv")


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

OUT = Path("/kaggle/working/CAFRNet_FINAL_MODEL_COMPARISON")

df = pd.read_csv(OUT / "all_models_robustness_comparison.csv")
class_df = pd.read_csv(OUT / "all_models_per_class_metrics.csv")

# ------------------------------------------------
# 1. Clean accuracy comparison
# ------------------------------------------------
clean = df[df.condition == "clean"].copy()
clean = clean.sort_values("accuracy", ascending=False)

plt.figure(figsize=(11, 6))
plt.bar(clean.model, clean.accuracy * 100)
plt.ylabel("Accuracy (%)")
plt.title("Clean Validation Accuracy: Six Models")
plt.xticks(rotation=25, ha="right")
plt.ylim(90, 100.2)
plt.tight_layout()
plt.savefig(OUT / "final_clean_accuracy_comparison.png", dpi=300)
plt.close()

# ------------------------------------------------
# 2. Accuracy across all six conditions
# ------------------------------------------------
pivot = df.pivot(index="model", columns="condition", values="accuracy") * 100

condition_order = [
    "clean", "noise_sigma20", "blur_radius2",
    "jpeg_quality20", "brightness_050", "brightness_150"
]
pivot = pivot[condition_order]

ax = pivot.T.plot(
    kind="bar", figsize=(14, 7), width=0.82
)
ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Evaluation condition")
ax.set_title("Robustness Comparison Across Six Conditions")
ax.set_ylim(90, 100.2)
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT / "final_robustness_comparison.png", dpi=300)
plt.close()

# ------------------------------------------------
# 3. Ablation comparison
# ------------------------------------------------
ablation_names = [
    "CAFRNet_Full",
    "CAFRNet_No_Edge",
    "CAFRNet_No_Semantic_Fusion",
    "CAFRNet_No_Attention"
]
abl = df[df.model.isin(ablation_names)].copy()

abl_pivot = abl.pivot(
    index="condition", columns="model", values="accuracy"
) * 100
abl_pivot = abl_pivot.loc[condition_order]

ax = abl_pivot.plot(kind="bar", figsize=(13, 7), width=0.82)
ax.set_ylabel("Accuracy (%)")
ax.set_title("CAFR-Net Component Ablation Study")
ax.legend(title="Configuration", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUT / "final_ablation_comparison.png", dpi=300)
plt.close()

# ------------------------------------------------
# 4. Macro-F1 comparison
# ------------------------------------------------
f1_pivot = df.pivot(
    index="model", columns="condition", values="macro_f1"
)[condition_order]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(f1_pivot.values, aspect="auto")
ax.set_xticks(range(len(condition_order)))
ax.set_xticklabels(condition_order, rotation=25, ha="right")
ax.set_yticks(range(len(f1_pivot.index)))
ax.set_yticklabels(f1_pivot.index)
ax.set_title("Macro-F1 Across Models and Conditions")
fig.colorbar(im, ax=ax, label="Macro-F1")
plt.tight_layout()
plt.savefig(OUT / "final_macro_f1_heatmap.png", dpi=300)
plt.close()

# ------------------------------------------------
# 5. Average corruption performance and ablation deltas
# ------------------------------------------------
corruptions = [c for c in condition_order if c != "clean"]

average_corruption = (
    df[df.condition.isin(corruptions)]
    .groupby("model")
    .agg(
        average_corruption_accuracy=("accuracy", "mean"),
        average_corruption_macro_f1=("macro_f1", "mean")
    )
    .reset_index()
)

clean_metrics = clean[
    ["model", "accuracy", "macro_f1"]
].rename(columns={
    "accuracy": "clean_accuracy",
    "macro_f1": "clean_macro_f1"
})

comparison = clean_metrics.merge(average_corruption, on="model")
comparison = comparison.sort_values(
    "average_corruption_accuracy", ascending=False
)

comparison.to_csv(OUT / "final_model_comparison_table.csv", index=False)
abl_pivot.to_csv(OUT / "final_ablation_accuracy_table.csv")

full = df[df.model == "CAFRNet_Full"].set_index("condition")
ablation_rows = []

for model_name in ablation_names[1:]:
    variant = df[df.model == model_name].set_index("condition")
    for condition in condition_order:
        ablation_rows.append({
            "variant": model_name,
            "condition": condition,
            "full_accuracy": full.loc[condition, "accuracy"],
            "variant_accuracy": variant.loc[condition, "accuracy"],
            "difference_percentage_points": (
                variant.loc[condition, "accuracy"]
                - full.loc[condition, "accuracy"]
            ) * 100
        })

ablation_delta = pd.DataFrame(ablation_rows)
ablation_delta.to_csv(OUT / "ablation_delta_vs_full.csv", index=False)

# ------------------------------------------------
# 6. Consolidated final research summary
# ------------------------------------------------
best_clean = clean.iloc[0]
best_corruption = comparison.iloc[0]

summary = {
    "project": "CAFR-Net v2",
    "dataset": "Imagenette2-320",
    "official_validation_images_per_condition": 3925,
    "number_of_models": 6,
    "number_of_conditions": 6,
    "total_image_condition_evaluations": 6 * 6 * 3925,
    "best_clean_model": best_clean["model"],
    "best_clean_accuracy": float(best_clean["accuracy"]),
    "best_clean_macro_f1": float(best_clean["macro_f1"]),
    "best_average_corruption_model": best_corruption["model"],
    "best_average_corruption_accuracy": float(
        best_corruption["average_corruption_accuracy"]
    ),
    "cafrnet_full_clean_accuracy": float(
        full.loc["clean", "accuracy"]
    ),
    "cafrnet_full_corruption_accuracy": {
        c: float(full.loc[c, "accuracy"]) for c in corruptions
    },
    "evaluation_conditions": condition_order,
    "interpretation": [
        "ConvNeXt-Tiny achieved the highest clean accuracy.",
        "CAFR-Net v2 outperformed ResNet-18 across all six evaluated conditions.",
        "The full CAFR-Net configuration showed stronger blur and noise performance than its component-removal variants.",
        "Performance differences must be interpreted in light of the warm-started, shorter ablation training budget."
    ],
    "limitations": [
        "Corruption evaluations use controlled transformations of the same official validation images.",
        "These results do not establish generalization to external datasets or real-world domain shifts.",
        "Ablations were warm-started from the full-model checkpoint and fine-tuned for 10 epochs, unlike the 30-epoch full-model training."
    ]
}

with open(OUT / "final_research_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))

print("\nABLATION DIFFERENCES VS FULL MODEL (percentage points)")
print(ablation_delta.to_string(index=False))

print("\nFINAL FIGURES AND TABLES CREATED")
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.suffix.lower() in {".png", ".csv", ".json"}:
        print(p.name)

print("\nFIGURE AND ANALYSIS GENERATION COMPLETED.")


In [ ]:

from pathlib import Path
import shutil, zipfile, hashlib, json, os
from datetime import datetime

ROOT = Path("/kaggle/working")
PACKAGE = ROOT / "CAFRNet_FINAL_RESEARCH_PACKAGE"
ZIP_PATH = ROOT / "CAFRNet_FINAL_RESEARCH_PACKAGE.zip"

if PACKAGE.exists():
    shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True)

# -------------------------------------------------------
# 1. Collect research folders and checkpoints
# -------------------------------------------------------
sources = {
    "source": ROOT / "CAFRNet_source",
    "final_model_comparison": ROOT / "CAFRNet_FINAL_MODEL_COMPARISON",
    "final_research_analysis": ROOT / "CAFRNet_continuation" / "FINAL_RESEARCH_ANALYSIS",
    "independent_test_results": ROOT / "CAFRNet_continuation" / "independent_test_results",
}

for dest_name, src in sources.items():
    if src.exists():
        shutil.copytree(src, PACKAGE / dest_name)
        print("Included:", dest_name)
    else:
        print("WARNING - not found:", src)

# Full CAFR-Net checkpoints
full_src = ROOT / "CAFRNet_continuation" / "cafrnet_seed42_continuation"
full_dest = PACKAGE / "checkpoints" / "CAFRNet_Full"
full_dest.mkdir(parents=True, exist_ok=True)

for filename in ["best_model.pt", "latest_model.pt"]:
    src = full_src / filename
    if src.exists():
        shutil.copy2(src, full_dest / filename)

# Ablation checkpoints
for name in ["no_edge", "no_semantic_fusion", "no_attention"]:
    src_dir = ROOT / "CAFRNet_ABLATIONS" / name
    dest_dir = PACKAGE / "checkpoints" / name
    dest_dir.mkdir(parents=True, exist_ok=True)

    for filename in ["best_model.pt", "latest_model.pt"]:
        src = src_dir / filename
        if src.exists():
            shutil.copy2(src, dest_dir / filename)

# Baseline checkpoints
for name in ["resnet18", "convnext_tiny"]:
    src_dir = ROOT / "CAFRNet_BASELINES" / name
    dest_dir = PACKAGE / "checkpoints" / name
    dest_dir.mkdir(parents=True, exist_ok=True)

    for filename in ["best_model.pt", "latest_model.pt"]:
        src = src_dir / filename
        if src.exists():
            shutil.copy2(src, dest_dir / filename)

# -------------------------------------------------------
# 2. Write final experiment documentation
# -------------------------------------------------------
readme = """
# CAFR-Net v2 — Final Research Experiment Package

## Dataset
Imagenette2-320, 10 classes.

The main model and baselines were trained using the reconstructed
seed-42 training/validation split. Final comparative evaluation used
the official validation directory containing 3,925 images.

## Model experiments
1. CAFR-Net v2 full configuration
2. CAFR-Net without edge refinement
3. CAFR-Net without semantic fusion
4. CAFR-Net without attention
5. ResNet-18 pretrained baseline
6. ConvNeXt-Tiny pretrained baseline

## Clean official validation results

| Model | Accuracy |
|---|---:|
| ConvNeXt-Tiny | 99.8981% |
| CAFR-Net v2 Full | 99.7962% |
| CAFR-Net No Semantic Fusion | 99.7962% |
| CAFR-Net No Edge | 99.7707% |
| CAFR-Net No Attention | 99.6943% |
| ResNet-18 | 97.9873% |

## Average corruption accuracy

| Model | Average accuracy |
|---|---:|
| ConvNeXt-Tiny | 99.3987% |
| CAFR-Net v2 Full | 99.2611% |
| CAFR-Net No Semantic Fusion | 99.1745% |
| CAFR-Net No Edge | 99.0675% |
| CAFR-Net No Attention | 98.7771% |
| ResNet-18 | 95.7911% |

Average corruption accuracy is the arithmetic mean across five
conditions, excluding clean images.

## Robustness conditions
- Gaussian noise, sigma 20
- Gaussian blur, radius 2
- JPEG compression, quality 20
- Brightness factor 0.5
- Brightness factor 1.5

Six models x six conditions x 3,925 images = 141,300
image-condition predictions.

## Ablation protocol qualification
The three ablation variants were initialized from the full-model
epoch-30 checkpoint and fine-tuned for 10 epochs. The full model
was trained for 30 epochs. Therefore, the ablation comparison is
a warm-started component-removal study with a different training
budget, not an equal-budget from-scratch comparison.

## Interpretation and limitations
- ConvNeXt-Tiny achieved the highest clean accuracy and highest
  average corruption accuracy in this evaluation.
- CAFR-Net v2 outperformed ResNet-18 across all six conditions.
- The full CAFR-Net showed better blur and noise accuracy than
  its component-removal variants.
- These controlled corruption tests use the same official
  validation images. They are not external-dataset evaluations.
- Results do not establish generalization to arbitrary unseen
  datasets, domains, or real-world distribution shifts.
- The official validation directory is used for comparative
  evaluation; it should not be described as an independent
  external test dataset.

## Reproducibility
The package includes model checkpoints, source code, predictions,
per-class metrics, confusion matrices, robustness results,
comparison tables, figures, and evaluation metadata.

Generated: """ + datetime.now().isoformat(timespec="seconds") + """

"""

(PACKAGE / "FINAL_RESEARCH_README.md").write_text(
    readme, encoding="utf-8"
)

# -------------------------------------------------------
# 3. Create SHA-256 manifest
# -------------------------------------------------------
manifest = []
for p in sorted(PACKAGE.rglob("*")):
    if p.is_file() and p.name != "SHA256_MANIFEST.json":
        digest = hashlib.sha256()
        with open(p, "rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(block)

        manifest.append({
            "file": str(p.relative_to(PACKAGE)),
            "size_bytes": p.stat().st_size,
            "sha256": digest.hexdigest()
        })

(PACKAGE / "SHA256_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

# -------------------------------------------------------
# 4. Create ZIP and verify integrity
# -------------------------------------------------------
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_STORED
) as z:
    for p in sorted(PACKAGE.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(PACKAGE.parent))

print("\nVerifying ZIP integrity...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad_file = z.testzip()
    entries = z.namelist()

assert bad_file is None, f"ZIP integrity failed: {bad_file}"

print("\n" + "=" * 65)
print("FINAL RESEARCH PACKAGE VERIFIED")
print("=" * 65)
print("ZIP:", ZIP_PATH)
print("Size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))
print("Files in package:", len(entries))
print("Integrity:", "PASS")
print("Manifest entries:", len(manifest))

print("\nCheckpoint files:")
for p in sorted((PACKAGE / "checkpoints").rglob("*.pt")):
    print(p.relative_to(PACKAGE), round(p.stat().st_size / (1024**2), 2), "MB")

print("\nFINAL PACKAGE READY")


In [ ]:

from pathlib import Path
import zipfile, re, os

ZIP_PATH = Path("/kaggle/working/CAFRNet_FINAL_RESEARCH_PACKAGE.zip")

print("=" * 65)
print("CAFR-Net v2 | QUICK RESEARCH PACKAGE AUDIT")
print("=" * 65)

if not ZIP_PATH.exists():
    print("ERROR: Research ZIP not found!")
    print("Files in /kaggle/working containing CAFR:")
    for p in Path("/kaggle/working").glob("*CAFR*"):
        print(" ", p)
    raise FileNotFoundError(ZIP_PATH)

print(f"\nZIP size: {ZIP_PATH.stat().st_size / (1024**3):.2f} GB")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad = z.testzip()
    print("ZIP integrity:", "PASS" if bad is None else f"FAIL: {bad}")

    names = z.namelist()
    print("Total entries:", len(names))

    # Locate important source and result files
    keywords = [
        "cafr_net.py", "train_architecture.py",
        "independent_test_metrics.json",
        "all_models_robustness_comparison.csv",
        "evaluation_metadata.json",
        "final_research_summary.json",
        "classification_report.csv",
        "README"
    ]

    print("\n--- IMPORTANT FILES ---")
    for keyword in keywords:
        matches = [n for n in names if keyword.lower() in n.lower()]
        print(f"\n{keyword}: {len(matches)} match(es)")
        for m in matches[:5]:
            print(" ", m)

    # Inspect CAFR-Net source without extracting it
    source_matches = [
        n for n in names
        if n.lower().endswith("cafr_net.py")
        and "legacy" not in n.lower()
    ]

    print("\n--- SOBEL / DENORMALIZATION AUDIT ---")
    if source_matches:
        for source_name in source_matches:
            source = z.read(source_name).decode("utf-8", errors="replace")
            lines = source.splitlines()
            print("\nSOURCE:", source_name)

            patterns = [
                r"mean", r"std", r"denorm", r"sobel",
                r"grayscale", r"gray", r"edge"
            ]

            for i, line in enumerate(lines):
                if any(re.search(p, line, re.I) for p in patterns):
                    print(f"{i+1}: {line[:220]}")

    else:
        print("No non-legacy cafr_net.py found in ZIP.")

    # Search text artifacts for historical accuracy discrepancy
    print("\n--- HISTORICAL ACCURACY DISCREPANCY ---")
    targets = ["78.115", "90.403"]
    found = {t: [] for t in targets}

    for name in names:
        if name.lower().endswith((".csv", ".json", ".txt", ".md", ".log")):
            try:
                data = z.read(name).decode("utf-8", errors="ignore")
                for t in targets:
                    if t in data:
                        found[t].append(name)
            except Exception:
                pass

    for t, files in found.items():
        print(f"{t}% appears in {len(files)} file(s):")
        for f in files[:10]:
            print(" ", f)

    # Locate cartoonization artifacts
    print("\n--- CARTOONIZATION ARTIFACT SEARCH ---")
    terms = ["opencv", "animegan", "stable_diffusion",
             "cartoonized", "cartoon", "stylized"]
    for term in terms:
        matches = [n for n in names if term in n.lower()]
        print(f"{term}: {len(matches)} entries")
        for m in matches[:3]:
            print(" ", m)

print("\nAUDIT COMPLETE")
print("No checkpoint or source files were modified.")


In [ ]:

from pathlib import Path
import zipfile, os, cv2

ZIP_PATH = Path("/kaggle/working/CAFRNet_FINAL_RESEARCH_PACKAGE.zip")

# Original Imagenette dataset
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320"),
    Path("/kaggle/input/imagenette2-320"),
]

print("=" * 60)
print("STEP 2: DATA AND CHECKPOINT DISCOVERY")
print("=" * 60)

print("\n--- IMAGENETTE DATASET ---")
DATA_ROOT = None

for p in DATASET_CANDIDATES:
    if p.exists():
        DATA_ROOT = p
        print("FOUND:", p)
        print("Train directory:", (p / "train").exists())
        print("Validation directory:", (p / "val").exists())
        if (p / "val").exists():
            classes = [x for x in (p / "val").iterdir() if x.is_dir()]
            count = sum(
                1 for c in classes
                for f in c.iterdir()
                if f.is_file()
            )
            print("Validation classes:", len(classes))
            print("Validation images:", count)
        break

if DATA_ROOT is None:
    print("Dataset not found in the expected locations.")

print("\n--- OPENCV ---")
print("OpenCV version:", cv2.__version__)

print("\n--- CHECKPOINTS IN ZIP ---")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    checkpoint_files = [
        n for n in z.namelist()
        if n.lower().endswith((".pt", ".pth", ".ckpt"))
    ]

    print("Total checkpoint files:", len(checkpoint_files))
    for n in checkpoint_files:
        info = z.getinfo(n)
        print(f"{info.file_size / (1024**2):8.2f} MB | {n}")

print("\n--- EXISTING EXTRACTED CHECKPOINTS ---")
for root in [
    Path("/kaggle/working/CAFRNet_continuation"),
    Path("/kaggle/working/CAFRNet_ABLATIONS"),
    Path("/kaggle/working/CAFRNet_BASELINES"),
]:
    if root.exists():
        print("\n", root)
        for p in root.rglob("best_model.pt"):
            print(p)

print("\nDISCOVERY COMPLETE")


In [ ]:

from pathlib import Path
import zipfile, re, torch

roots = [
    Path("/kaggle/working/CAFRNet_source"),
    Path("/kaggle/working/CAFRNet_continuation"),
    Path("/kaggle/working/CAFRNet_FINAL_MODEL_COMPARISON"),
    Path("/kaggle/working/CAFRNet_ABLATIONS"),
    Path("/kaggle/working/CAFRNet_BASELINES"),
]

print("========== EXISTING EXPERIMENT SCRIPTS ==========")
for root in roots:
    if root.exists():
        print("\nROOT:", root)
        for p in root.rglob("*"):
            if p.is_file() and (
                p.suffix.lower() in [".py", ".ipynb", ".json", ".log"]
                or "eval" in p.name.lower()
            ):
                print(p)

print("\n========== MODEL SOURCE SIGNATURES ==========")
for p in [
    Path("/kaggle/working/CAFRNet_source/cafr_net.py"),
    Path("/kaggle/working/CAFRNet_source/train_architecture.py"),
]:
    if p.exists():
        print("\nFILE:", p)
        lines = p.read_text(errors="ignore").splitlines()
        for i, line in enumerate(lines):
            if re.search(r"^class |^def |^    def __init__|build_model|ConvNeXt", line):
                print(f"{i+1}: {line[:180]}")

print("\n========== CHECKPOINT METADATA ==========")
checkpoint_paths = {
    "CAFRNet_Full": Path("/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt"),
    "No_Edge": Path("/kaggle/working/CAFRNet_ABLATIONS/no_edge/best_model.pt"),
    "No_Semantic_Fusion": Path("/kaggle/working/CAFRNet_ABLATIONS/no_semantic_fusion/best_model.pt"),
    "No_Attention": Path("/kaggle/working/CAFRNet_ABLATIONS/no_attention/best_model.pt"),
    "ResNet18": Path("/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt"),
    "ConvNeXt_Tiny": Path("/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt"),
}

for name, path in checkpoint_paths.items():
    if path.exists():
        ck = torch.load(path, map_location="cpu", weights_only=False)
        print("\n", name, "|", path.name)
        if isinstance(ck, dict):
            print("Keys:", list(ck.keys())[:20])
            sd = ck.get("model_state_dict", ck.get("state_dict", ck))
            if isinstance(sd, dict):
                print("State dict tensors:", len(sd))
                print("First keys:", list(sd.keys())[:8])
    else:
        print("\nMISSING:", name, path)

print("\nDISCOVERY COMPLETE")


In [ ]:

from pathlib import Path
import torch, json

SRC = Path("/kaggle/working/CAFRNet_source")
CKPTS = {
    "CAFRNet_Full": "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
    "No_Edge": "/kaggle/working/CAFRNet_ABLATIONS/no_edge/best_model.pt",
    "No_Semantic_Fusion": "/kaggle/working/CAFRNet_ABLATIONS/no_semantic_fusion/best_model.pt",
    "No_Attention": "/kaggle/working/CAFRNet_ABLATIONS/no_attention/best_model.pt",
    "ResNet18": "/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt",
    "ConvNeXt_Tiny": "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
}

print("========== TRAINING MODEL FACTORY ==========")
p = SRC / "train_architecture.py"
lines = p.read_text().splitlines()
for start, end in [(65, 130), (150, 225)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start-1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

print("\n========== CAFR MODEL FORWARD ==========")
p = SRC / "cafr_net.py"
lines = p.read_text().splitlines()
for start, end in [(123, 218), (243, 260)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start-1, min(end, len(lines))):
        print(f"{i+1}: {lines[i]}")

print("\n========== CHECKPOINT CONFIGURATIONS ==========")
for name, path in CKPTS.items():
    ck = torch.load(path, map_location="cpu", weights_only=False)
    print("\n", name)
    print("Architecture:", ck.get("architecture"))
    print("Config:", ck.get("model_config"))
    print("Classes:", ck.get("classes"))
    print("Seed:", ck.get("seed"), "| Epoch:", ck.get("epoch"))
    print("Val accuracy:", ck.get("val_accuracy"))
    print("Metadata:", ck.get("metadata"))

print("\n========== EVALUATION METADATA ==========")
meta = Path("/kaggle/working/CAFRNet_FINAL_MODEL_COMPARISON/evaluation_metadata.json")
if meta.exists():
    print(meta.read_text()[:12000])

print("\nSTEP 4 COMPLETE")


In [ ]:

import os, sys, csv, time, json, gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from PIL import Image

# ---------------- CONFIG ----------------
SRC = Path("/kaggle/working/CAFRNet_source")
DATA = Path("/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320/val")
OUT = Path("/kaggle/working/CAFRNet_CARTOON_EVALUATION")
OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH = 16
WORKERS = 2

sys.path.insert(0, str(SRC))
from train_architecture import build_model

CHECKPOINTS = {
    "CAFRNet_Full": (
        "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
        "cafrnet_v2", "full"),
    "CAFRNet_No_Edge": (
        "/kaggle/working/CAFRNet_ABLATIONS/no_edge/best_model.pt",
        "cafrnet_v2", "no_edge"),
    "CAFRNet_No_Semantic_Fusion": (
        "/kaggle/working/CAFRNet_ABLATIONS/no_semantic_fusion/best_model.pt",
        "cafrnet_v2", "no_semantic_fusion"),
    "CAFRNet_No_Attention": (
        "/kaggle/working/CAFRNet_ABLATIONS/no_attention/best_model.pt",
        "cafrnet_v2", "no_attention"),
    "ResNet18": (
        "/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt",
        "resnet18", "full"),
    "ConvNeXt_Tiny": (
        "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
        "convnext_tiny", "full"),
}

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# ---------------- DATASET ----------------
base = datasets.ImageFolder(str(DATA))
classes = base.classes
samples = base.samples

class CartoonDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
        self.tensor = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD)
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        rgb = np.array(Image.open(path).convert("RGB"))
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

        # OpenCV edge-preserving cartoon stylization
        cartoon_bgr = cv2.stylization(
            bgr, sigma_s=60, sigma_r=0.45
        )
        cartoon_rgb = cv2.cvtColor(
            cartoon_bgr, cv2.COLOR_BGR2RGB
        )

        image = Image.fromarray(cartoon_rgb)
        return self.tensor(image), label, idx

loader = DataLoader(
    CartoonDataset(samples),
    batch_size=BATCH,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Device:", DEVICE)
print("Images:", len(samples))
print("Classes:", classes)
print("Output:", OUT)

# ---------------- EVALUATION ----------------
all_rows = []
summary = []

for model_name, (checkpoint_path, arch, ablation) in CHECKPOINTS.items():
    start = time.time()
    print(f"\nEvaluating: {model_name}", flush=True)

    ckpt = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )

    assert ckpt["classes"] == classes, (
        f"Class order mismatch for {model_name}"
    )

    model, _, _, _ = build_model(
        arch, num_classes=len(classes), ablation=ablation
    )
    model.load_state_dict(ckpt["model"], strict=True)
    model.to(DEVICE).eval()

    correct = 0
    total = 0
    model_rows = []

    with torch.inference_mode():
        for images, labels, indices in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if DEVICE.type == "cuda":
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(images)
            else:
                logits = model(images)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(logits.float(), dim=1)
            conf = probs.max(dim=1).values

            for idx, y, pred, c in zip(
                indices.tolist(),
                labels.cpu().tolist(),
                preds.cpu().tolist(),
                conf.cpu().tolist()
            ):
                row = {
                    "model": model_name,
                    "condition": "opencv_cartoon_sigma_s60_sigma_r045",
                    "image_index": idx,
                    "image_path": samples[idx][0],
                    "true_class": classes[y],
                    "predicted_class": classes[pred],
                    "correct": int(y == pred),
                    "confidence": float(c)
                }
                model_rows.append(row)

    accuracy = correct / total
    elapsed = time.time() - start

    all_rows.extend(model_rows)
    summary.append({
        "model": model_name,
        "condition": "opencv_cartoon_sigma_s60_sigma_r045",
        "correct": correct,
        "total": total,
        "accuracy": accuracy,
        "accuracy_percent": accuracy * 100,
        "seconds": elapsed
    })

    print(
        f"{model_name}: {correct}/{total} "
        f"= {accuracy*100:.4f}% | {elapsed/60:.1f} min",
        flush=True
    )

    del model, ckpt
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ---------------- SAVE ----------------
pred_path = OUT / "opencv_cartoon_all_predictions.csv"
summary_path = OUT / "opencv_cartoon_model_comparison.csv"

pd.DataFrame(all_rows).to_csv(pred_path, index=False)
pd.DataFrame(summary).sort_values(
    "accuracy", ascending=False
).to_csv(summary_path, index=False)

print("\n========== CARTOONIZATION RESULTS ==========")
print(pd.DataFrame(summary).sort_values(
    "accuracy", ascending=False
).to_string(index=False))

print("\nSaved predictions:", pred_path)
print("Saved comparison:", summary_path)

with open(OUT / "metadata.json", "w") as f:
    json.dump({
        "dataset": str(DATA),
        "images": len(samples),
        "classes": classes,
        "transformation": "cv2.stylization",
        "sigma_s": 60,
        "sigma_r": 0.45,
        "models": list(CHECKPOINTS),
        "device": str(DEVICE),
        "note": "Inference-only evaluation; no model retraining."
    }, f, indent=2)

print("\nEXPERIMENT COMPLETE")


In [ ]:
import pandas as pd
from pathlib import Path

files = [
    Path("/kaggle/working/CAFRNet_CARTOON_EVALUATION/opencv_cartoon_all_predictions.csv"),
    Path("/kaggle/working/CAFRNet_FINAL_MODEL_COMPARISON/all_models_predictions.csv"),
]

for f in files:
    print("\nFILE:", f.name, "| Exists:", f.exists())
    if f.exists():
        df = pd.read_csv(f)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import binomtest

BASE = Path("/kaggle/working/CAFRNet_FINAL_MODEL_COMPARISON")
CARTOON = Path("/kaggle/working/CAFRNet_CARTOON_EVALUATION")

# Load existing predictions
df1 = pd.read_csv(BASE / "all_models_predictions.csv")
df2 = pd.read_csv(CARTOON / "opencv_cartoon_all_predictions.csv")

# Standardize column names
df1 = df1.rename(columns={"image": "image_key"})
df2 = df2.rename(columns={"image_path": "image_key"})

df = pd.concat([df1, df2], ignore_index=True)
df["correct"] = (
    df["correct"].astype(str).str.lower().isin(["true", "1"])
)

models = ["CAFRNet_Full", "ConvNeXt_Tiny"]
rng = np.random.default_rng(42)
results = []

for condition in df["condition"].unique():
    part = df[
        (df["condition"] == condition) &
        (df["model"].isin(models))
    ]

    paired = part.pivot(
        index="image_key",
        columns="model",
        values="correct"
    ).dropna()

    a = paired["CAFRNet_Full"].to_numpy(dtype=bool)
    b = paired["ConvNeXt_Tiny"].to_numpy(dtype=bool)

    n = len(paired)
    cafr_correct = int(a.sum())
    conv_correct = int(b.sum())

    # Discordant paired outcomes
    cafr_only = int(np.sum(a & ~b))
    conv_only = int(np.sum(~a & b))
    discordant = cafr_only + conv_only

    # Exact two-sided McNemar test
    p_value = (
        binomtest(cafr_only, discordant, 0.5).pvalue
        if discordant > 0 else 1.0
    )

    # Paired bootstrap confidence interval
    differences = np.empty(3000)

    for i in range(3000):
        idx = rng.integers(0, n, size=n)
        differences[i] = (a[idx].mean() - b[idx].mean()) * 100

    ci_low, ci_high = np.percentile(differences, [2.5, 97.5])

    results.append({
        "condition": condition,
        "images": n,
        "CAFR_accuracy_%": cafr_correct / n * 100,
        "ConvNeXt_accuracy_%": conv_correct / n * 100,
        "difference_CAFR_minus_ConvNeXt_pp":
            (cafr_correct - conv_correct) / n * 100,
        "CAFR_only_correct": cafr_only,
        "ConvNeXt_only_correct": conv_only,
        "discordant_pairs": discordant,
        "McNemar_p": p_value,
        "bootstrap_95CI_low_pp": ci_low,
        "bootstrap_95CI_high_pp": ci_high
    })

result = pd.DataFrame(results)

# Holm correction for multiple comparisons
p = result["McNemar_p"].to_numpy()
order = np.argsort(p)
adjusted = np.empty(len(p))
running_max = 0

for rank, idx in enumerate(order):
    adjusted[idx] = min(1.0, (len(p) - rank) * p[idx])
    running_max = max(running_max, adjusted[idx])
    adjusted[idx] = running_max

result["Holm_adjusted_p"] = adjusted
result["significant_0.05"] = result["Holm_adjusted_p"] < 0.05

out = CARTOON / "CAFR_vs_ConvNeXt_paired_statistics.csv"
result.to_csv(out, index=False)

print("\n========== PAIRED STATISTICAL RESULTS ==========")
print(result.round(5).to_string(index=False))
print("\nSaved:", out)
print("\nConditions tested:", len(result))
print("All paired image counts:", result["images"].tolist())

In [ ]:
import os, sys, time, cv2, torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

sys.path.insert(0, "/kaggle/working/CAFRNet_source")
from train_architecture import build_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = "/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320/val"
OUT = Path("/kaggle/working/CAFRNet_SEVERE_EVALUATION")
OUT.mkdir(exist_ok=True)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

base = ImageFolder(DATA)
classes = base.classes
samples = base.samples

CHECKPOINTS = {
    "CAFRNet_Full": (
        "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
        "cafrnet_v2", None
    ),
    "ConvNeXt_Tiny": (
        "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
        "convnext_tiny", None
    )
}

CONDITIONS = ["noise_sigma40", "blur_severe", "jpeg_quality10"]

class SevereDataset(Dataset):
    def __init__(self, condition):
        self.condition = condition

    def __len__(self):
        return len(samples)

    def __getitem__(self, idx):
        path, label = samples[idx]
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.condition == "noise_sigma40":
            rng = np.random.default_rng(idx + 4000)
            noise = rng.normal(0, 40, img.shape)
            img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        elif self.condition == "blur_severe":
            img = cv2.GaussianBlur(img, (9, 9), 0)

        elif self.condition == "jpeg_quality10":
            bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            ok, encoded = cv2.imencode(
                ".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 10]
            )
            img = cv2.cvtColor(cv2.imdecode(encoded, 1), cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        img = (img - MEAN) / STD
        img = torch.from_numpy(img.transpose(2, 0, 1)).float()

        return img, label, idx

results = []

for model_name, (ckpt_path, arch, ablation) in CHECKPOINTS.items():
    print(f"\nLoading {model_name}...")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    model, _, _, _ = build_model(
        arch, num_classes=len(classes), ablation=ablation
    )
    model.load_state_dict(ckpt["model"], strict=True)
    model = model.to(DEVICE).eval()

    for condition in CONDITIONS:
        loader = DataLoader(
            SevereDataset(condition),
            batch_size=64,
            shuffle=False,
            num_workers=2,
            pin_memory=(DEVICE == "cuda")
        )

        correct = total = 0
        start = time.time()

        with torch.inference_mode():
            for x, y, _ in loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=(DEVICE == "cuda")
                ):
                    logits = model(x)
                    if isinstance(logits, (tuple, list)):
                        logits = logits[0]
                    if isinstance(logits, dict):
                        logits = logits.get("logits", next(iter(logits.values())))

                correct += (logits.argmax(1) == y).sum().item()
                total += y.size(0)

        elapsed = time.time() - start
        accuracy = correct / total * 100

        results.append({
            "model": model_name,
            "condition": condition,
            "correct": correct,
            "total": total,
            "accuracy_percent": accuracy,
            "seconds": elapsed
        })

        print(
            f"{model_name} | {condition}: "
            f"{correct}/{total} = {accuracy:.4f}% | {elapsed/60:.1f} min"
        )

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

result = pd.DataFrame(results)
result.to_csv(OUT / "severe_corruption_comparison.csv", index=False)

print("\n========== SEVERE CORRUPTION RESULTS ==========")
print(result.to_string(index=False))
print("\nSaved:", OUT / "severe_corruption_comparison.csv")

In [ ]:
import sys, time, cv2, torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

sys.path.insert(0, "/kaggle/working/CAFRNet_source")
from train_architecture import build_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = "/kaggle/input/datasets/xbinchen/imagenette2-320/imagenette2-320/val"
OUT = Path("/kaggle/working/CAFRNet_SEVERE_EVALUATION")
OUT.mkdir(exist_ok=True)

base = ImageFolder(DATA)
samples, classes = base.samples, base.classes

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class SevereDataset(Dataset):
    def __init__(self, condition):
        self.condition = condition

    def __len__(self):
        return len(samples)

    def __getitem__(self, idx):
        path, label = samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

        if self.condition == "noise_sigma40":
            rng = np.random.default_rng(idx + 4000)
            img = np.clip(
                img.astype(np.float32) +
                rng.normal(0, 40, img.shape), 0, 255
            ).astype(np.uint8)
        elif self.condition == "blur_severe":
            img = cv2.GaussianBlur(img, (9, 9), 0)
        elif self.condition == "jpeg_quality10":
            bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            _, enc = cv2.imencode(
                ".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 10]
            )
            img = cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (224, 224)).astype(np.float32) / 255
        img = (img - MEAN) / STD
        return torch.from_numpy(img.transpose(2, 0, 1)).float(), label

ckpt_path = (
    "/kaggle/working/CAFRNet_BASELINES/"
    "convnext_tiny/best_model.pt"
)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

# Baseline builder requires ablation="full" in this source implementation.
model, _, _, _ = build_model(
    "convnext_tiny",
    num_classes=len(classes),
    ablation="full"
)
model.load_state_dict(ckpt["model"], strict=True)
model = model.to(DEVICE).eval()

results = []

for condition in ["noise_sigma40", "blur_severe", "jpeg_quality10"]:
    loader = DataLoader(
        SevereDataset(condition),
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE == "cuda")
    )

    correct = total = 0
    start = time.time()

    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE == "cuda")
            ):
                logits = model(x)
                if isinstance(logits, (tuple, list)):
                    logits = logits[0]
                if isinstance(logits, dict):
                    logits = logits.get("logits", next(iter(logits.values())))

            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)

    elapsed = time.time() - start
    results.append({
        "model": "ConvNeXt_Tiny",
        "condition": condition,
        "correct": correct,
        "total": total,
        "accuracy_percent": correct / total * 100,
        "seconds": elapsed
    })

    print(
        f"{condition}: {correct}/{total} = "
        f"{correct/total*100:.4f}% | {elapsed/60:.2f} min"
    )

del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

new = pd.DataFrame(results)
new.to_csv(OUT / "convnext_severe_results.csv", index=False)

# Combine with the already completed CAFR-Net results
old = pd.read_csv(OUT / "severe_corruption_comparison.csv")
combined = pd.concat([old, new], ignore_index=True)
combined.to_csv(OUT / "severe_corruption_comparison.csv", index=False)

print("\n========== COMBINED SEVERE RESULTS ==========")
print(combined.to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("/kaggle/working/CAFRNet_SEVERE_EVALUATION")
OUT.mkdir(exist_ok=True)

rows = [
    ["CAFRNet_Full", "noise_sigma40", 3889, 3925, 99.0828, 30],
    ["CAFRNet_Full", "blur_severe", 3914, 3925, 99.7197, 12],
    ["CAFRNet_Full", "jpeg_quality10", 3903, 3925, 99.4395, 12],
    ["ConvNeXt_Tiny", "noise_sigma40", 3886, 3925, 99.0064, 29.4],
    ["ConvNeXt_Tiny", "blur_severe", 3919, 3925, 99.8471, 13.8],
    ["ConvNeXt_Tiny", "jpeg_quality10", 3908, 3925, 99.5669, 15],
]

df = pd.DataFrame(rows, columns=[
    "model", "condition", "correct", "total",
    "accuracy_percent", "seconds_approx"
])

path = OUT / "severe_corruption_comparison.csv"
df.to_csv(path, index=False)

print(df.to_string(index=False))
print("\nSaved:", path)

In [ ]:
import sys, time, torch, pandas as pd
from pathlib import Path

sys.path.insert(0, "/kaggle/working/CAFRNet_source")
from train_architecture import build_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINTS = {
    "CAFRNet_Full": (
        "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
        "cafrnet_v2"
    ),
    "ConvNeXt_Tiny": (
        "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt",
        "convnext_tiny"
    ),
    "ResNet18": (
        "/kaggle/working/CAFRNet_BASELINES/resnet18/best_model.pt",
        "resnet18"
    )
}

results = []

for name, (path, arch) in CHECKPOINTS.items():
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    model, _, _, _ = build_model(
        arch,
        num_classes=len(ckpt["classes"]),
        ablation="full"
    )
    model.load_state_dict(ckpt["model"], strict=True)
    model = model.to(DEVICE).eval()

    params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\nBenchmarking {name} | Parameters: {params:,}")

    batch = torch.randn(1, 3, 224, 224, device=DEVICE)

    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        for _ in range(20):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE == "cuda")
            ):
                model(batch)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        iterations = 100

        for _ in range(iterations):
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE == "cuda")
            ):
                model(batch)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

    latency = elapsed / iterations * 1000
    throughput = iterations / elapsed
    memory = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if DEVICE == "cuda" else 0
    )

    results.append({
        "model": name,
        "parameters": params,
        "trainable_parameters": trainable,
        "batch1_latency_ms": latency,
        "throughput_images_sec": throughput,
        "peak_gpu_memory_MB": memory
    })

    print(f"Latency: {latency:.3f} ms")
    print(f"Throughput: {throughput:.2f} images/sec")
    print(f"Peak GPU memory: {memory:.2f} MB")

    del model, batch
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

result = pd.DataFrame(results)
out = Path("/kaggle/working/CAFRNet_EFFICIENCY")
out.mkdir(exist_ok=True)

result.to_csv(out / "efficiency_comparison.csv", index=False)

print("\n========== FINAL EFFICIENCY COMPARISON ==========")
print(result.to_string(index=False))
print("\nSaved:", out / "efficiency_comparison.csv")

In [ ]:
import os, json, shutil, zipfile, hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime

WORK = Path("/kaggle/working")
SOURCE_ZIP = WORK / "CAFRNet_FINAL_RESEARCH_PACKAGE.zip"
STAGING = WORK / "CAFRNet_PACKAGE_UPDATE"
NEW_ZIP = WORK / "CAFRNet_FINAL_RESEARCH_PACKAGE_UPDATED.zip"

# 1. Verify original package
assert SOURCE_ZIP.exists(), f"Original ZIP not found: {SOURCE_ZIP}"

if STAGING.exists():
    shutil.rmtree(STAGING)
STAGING.mkdir(parents=True)

with zipfile.ZipFile(SOURCE_ZIP, "r") as z:
    bad = z.testzip()
    assert bad is None, f"Original ZIP failed integrity check: {bad}"
    z.extractall(STAGING)

# Find the extracted package root
roots = [p for p in STAGING.iterdir() if p.is_dir()]
assert roots, "Could not find extracted package directory."
PACKAGE = roots[0]

print("Original package extracted:", PACKAGE)

# 2. Copy all newly generated experimental artifacts
ARTIFACTS = {
    "cartoonization/opencv_cartoon_all_predictions.csv":
        WORK / "CAFRNet_CARTOON_EVALUATION/opencv_cartoon_all_predictions.csv",

    "cartoonization/opencv_cartoon_model_comparison.csv":
        WORK / "CAFRNet_CARTOON_EVALUATION/opencv_cartoon_model_comparison.csv",

    "statistics/CAFR_vs_ConvNeXt_paired_statistics.csv":
        WORK / "CAFRNet_CARTOON_EVALUATION/CAFR_vs_ConvNeXt_paired_statistics.csv",

    "severe_corruptions/severe_corruption_comparison.csv":
        WORK / "CAFRNet_SEVERE_EVALUATION/severe_corruption_comparison.csv",

    "efficiency/efficiency_comparison.csv":
        WORK / "CAFRNet_EFFICIENCY/efficiency_comparison.csv",
}

added = []
missing = []

for relative, source in ARTIFACTS.items():
    if source.exists():
        destination = PACKAGE / "additional_experiments" / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        added.append(str(destination.relative_to(PACKAGE)))
        print("ADDED:", relative)
    else:
        missing.append(str(source))
        print("MISSING:", source)

assert len(added) == len(ARTIFACTS), (
    "Some result files are missing. Check the paths above before continuing."
)

# 3. Read results to create a structured summary
cartoon = pd.read_csv(
    PACKAGE / "additional_experiments/cartoonization/opencv_cartoon_model_comparison.csv"
)
stats = pd.read_csv(
    PACKAGE / "additional_experiments/statistics/CAFR_vs_ConvNeXt_paired_statistics.csv"
)
severe = pd.read_csv(
    PACKAGE / "additional_experiments/severe_corruptions/severe_corruption_comparison.csv"
)
efficiency = pd.read_csv(
    PACKAGE / "additional_experiments/efficiency/efficiency_comparison.csv"
)

summary = {
    "project": "CAFR-Net v2",
    "package_update_date": datetime.now().isoformat(timespec="seconds"),
    "dataset": "Imagenette2-320",
    "official_validation_images": 3925,
    "device_for_reported_gpu_experiments": "NVIDIA Tesla T4 (Kaggle)",
    "new_experiments": {
        "opencv_cartoonization": cartoon.to_dict(orient="records"),
        "paired_statistics": stats.to_dict(orient="records"),
        "severe_corruptions": severe.to_dict(orient="records"),
        "efficiency": efficiency.to_dict(orient="records")
    },
    "statistical_interpretation": {
        "comparison": "CAFRNet_Full versus ConvNeXt_Tiny",
        "conditions_tested": int(len(stats)),
        "holm_significant_conditions": int(
            (stats["Holm_adjusted_p"] < 0.05).sum()
        ),
        "interpretation": (
            "No tested condition showed a statistically significant "
            "difference after Holm correction at alpha=0.05."
        )
    },
    "limitations_and_pending_work": [
        "Historical OpenCV accuracy discrepancy (78.115% versus 90.403%) remains unresolved.",
        "AnimeGANv2 and Stable Diffusion cartoonization evaluations remain pending.",
        "Additional seeds 123 and 314 for CAFR-Net Full and ConvNeXt-Tiny remain pending.",
        "Matched ConvNeXt-Tiny evaluation on Oxford-IIIT Pet remains pending.",
        "Scratch-trained multi-seed ablations remain pending.",
        "ConvNeXt plus CBAM and ConvNeXt plus AugMix remain pending.",
        "Full ImageNet-C and PACS evaluations remain pending.",
        "The severe-corruption tests are controlled stress tests, not a full ImageNet-C benchmark.",
        "Efficiency measurements are specific to the reported Kaggle hardware and benchmark settings."
    ]
}

summary_path = PACKAGE / "additional_experiments" / "updated_experiment_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# 4. Create a human-readable research addendum
def markdown_table(df):
    return df.to_markdown(index=False)

report = f"""# CAFR-Net v2 — Additional Experimental Results

## Package update

Updated: {summary["package_update_date"]}

Dataset: Imagenette2-320 official validation set, 3,925 images.

## 1. OpenCV cartoonization

All six models were evaluated on OpenCV-stylized images using
sigma_s=60 and sigma_r=0.45.

{markdown_table(cartoon[["model", "correct", "total", "accuracy_percent"]])}

ConvNeXt-Tiny achieved the highest accuracy in this experiment.
CAFR-Net Full outperformed the three tested CAFR-Net ablations and ResNet18,
but did not outperform ConvNeXt-Tiny.

## 2. Paired statistical comparison

CAFR-Net Full versus ConvNeXt-Tiny was compared over seven conditions.
The analysis used exact McNemar tests, paired bootstrap confidence intervals
with 3,000 resamples, and Holm correction.

{markdown_table(stats)}

Number of conditions significant after Holm correction:
**{summary["statistical_interpretation"]["holm_significant_conditions"]} of {len(stats)}**.

No condition crossed the 0.05 adjusted significance threshold.

## 3. High-severity corruption tests

{markdown_table(severe)}

These are controlled severe stress tests:
Gaussian noise sigma=40, Gaussian blur kernel 9x9, and JPEG quality 10.
They are not a full ImageNet-C evaluation.

## 4. Inference efficiency

{markdown_table(efficiency)}

Measurements were obtained on the current Kaggle GPU environment using
batch size 1, 20 warm-up iterations and 100 timed iterations.
Peak GPU memory is the maximum allocated memory observed during the benchmark.
These measurements are hardware- and software-environment-specific.

## 5. Existing research results

The original research package and its existing standard-corruption,
clean-validation, baseline and ablation results have been preserved.
This update adds the new experiments; it does not replace historical results.

## 6. Outstanding work

- Investigate the historical OpenCV discrepancy: 78.115% versus 90.403%.
- Evaluate AnimeGANv2 and Stable Diffusion cartoonization domains.
- Run CAFR-Net Full and ConvNeXt-Tiny with seeds 123 and 314.
- Evaluate a matched ConvNeXt-Tiny baseline on Oxford-IIIT Pet.
- Consider scratch-trained multi-seed ablations.
- Consider ConvNeXt+CBAM and ConvNeXt+AugMix.
- Consider full ImageNet-C and PACS evaluations.

## 7. Research interpretation

The current evidence demonstrates strong performance by CAFR-Net Full on
the tested Imagenette conditions. ConvNeXt-Tiny remains slightly stronger
on the standard corruption mean and OpenCV cartoonization evaluation.
Paired statistical testing does not establish a significant difference
after multiple-comparison correction. CAFR-Net Full has higher measured
inference cost than ConvNeXt-Tiny and ResNet18 in the current benchmark.

These results should not be described as proof of general superiority
or as a completed cross-dataset generalization study.
"""

report_path = PACKAGE / "additional_experiments" / "RESEARCH_RESULTS_ADDENDUM.md"
report_path.write_text(report, encoding="utf-8")

# 5. Append a link to the main README without overwriting its content
readme_candidates = list(PACKAGE.rglob("README.md"))
if readme_candidates:
    readme = readme_candidates[0]
    old_text = readme.read_text(encoding="utf-8", errors="replace")
    marker = "## Additional experimental results — November 2026"

    if marker not in old_text:
        with readme.open("a", encoding="utf-8") as f:
            f.write(
                "\n\n" + marker +
                "\nNew cartoonization, paired statistical, severe-corruption "
                "and efficiency results are documented in "
                "`additional_experiments/RESEARCH_RESULTS_ADDENDUM.md`.\n"
            )
        print("README updated:", readme)
    else:
        print("README already contains update marker.")
else:
    print("WARNING: No README.md found in extracted package.")

# 6. Create a file manifest with SHA-256 checksums
manifest = []
for path in sorted(PACKAGE.rglob("*")):
    if path.is_file():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        manifest.append({
            "file": str(path.relative_to(PACKAGE)),
            "size_bytes": path.stat().st_size,
            "sha256": digest
        })

manifest_path = PACKAGE / "UPDATED_PACKAGE_MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# 7. Build the new ZIP
if NEW_ZIP.exists():
    NEW_ZIP.unlink()

with zipfile.ZipFile(
    NEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3
) as z:
    for path in PACKAGE.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(STAGING))

# 8. Verify the new ZIP
with zipfile.ZipFile(NEW_ZIP, "r") as z:
    bad = z.testzip()
    assert bad is None, f"ZIP integrity failure: {bad}"
    entries = z.namelist()

print("\n" + "=" * 60)
print("UPDATED RESEARCH PACKAGE CREATED")
print("=" * 60)
print("ZIP:", NEW_ZIP)
print("Size (GB):", round(NEW_ZIP.stat().st_size / (1024**3), 3))
print("ZIP entries:", len(entries))
print("Integrity:", "PASS")
print("New result files:", len(added))
print("Manifest:", manifest_path)
print("\nAdded artifacts:")
for item in added:
    print(" -", item)
print("\nNext: download the updated ZIP and push the updated research folder to GitHub.")

In [ ]:
from pathlib import Path

source = Path("/kaggle/working/CAFRNet_source")

for filename in ["train_architecture.py", "cafr_net.py"]:
    path = source / filename
    print("\n", "=" * 55)
    print(filename, "exists:", path.exists())

    if path.exists() and filename == "train_architecture.py":
        lines = path.read_text().splitlines()

        for i, line in enumerate(lines):
            if any(word in line.lower() for word in [
                "resume", "checkpoint", "start_epoch",
                "load_state_dict", "argparse"
            ]):
                start = max(0, i - 2)
                end = min(len(lines), i + 4)
                print(f"\n--- Lines {start+1}-{end} ---")
                for j in range(start, end):
                    print(f"{j+1}: {lines[j]}")

In [ ]:
from pathlib import Path
import torch

SRC = Path("/kaggle/working/CAFRNet_source/train_architecture.py")

lines = SRC.read_text().splitlines()

print("========== TRAINING ARGUMENTS ==========")
for i in range(153, 190):
    print(f"{i+1}: {lines[i]}")

print("\n========== AVAILABLE DATA DIRECTORIES ==========")
for p in Path("/kaggle/working").iterdir():
    if p.is_dir() and any(x in p.name.lower() for x in ["data", "split", "imagenette"]):
        print(p)

print("\n========== SEED-42 CHECKPOINT METADATA ==========")

checkpoints = {
    "CAFRNet": "/kaggle/working/CAFRNet_continuation/cafrnet_seed42_continuation/best_model.pt",
    "ConvNeXt": "/kaggle/working/CAFRNet_BASELINES/convnext_tiny/best_model.pt"
}

for name, path in checkpoints.items():
    p = Path(path)
    if not p.exists():
        print(name, "checkpoint missing:", path)
        continue

    ck = torch.load(p, map_location="cpu", weights_only=False)
    print("\n", name)
    print("Architecture:", ck.get("architecture"))
    print("Seed:", ck.get("seed"))
    print("Epoch:", ck.get("epoch"))
    print("Validation accuracy:", ck.get("val_accuracy"))
    print("Metadata:", ck.get("metadata"))

In [ ]:
from pathlib import Path
import torch

DATA = Path("/kaggle/working/CAFRNet_continuation/cafr_data")
SRC = Path("/kaggle/working/CAFRNet_source/train_architecture.py")

print("========== DATASET CHECK ==========")
for folder in ["train", "val"]:
    p = DATA / folder
    print(folder, "exists:", p.exists())

    if p.exists():
        classes = sorted(x.name for x in p.iterdir() if x.is_dir())
        count = sum(
            1 for c in classes for f in (p / c).rglob("*")
            if f.is_file()
        )
        print("Classes:", len(classes), "| Images:", count)

print("\n========== PRETRAINED WEIGHT REFERENCES ==========")
lines = SRC.read_text().splitlines()

for i, line in enumerate(lines):
    if any(x in line.lower() for x in [
        "pretrained", "weights_path", "weight_path",
        "convnext_tiny", "resnet18"
    ]):
        print(f"{i+1}: {line}")

print("\n========== GPU ==========")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Free memory (GB):",
          round(torch.cuda.mem_get_info()[0] / 1024**3, 2))

print("\n========== STORAGE ==========")
import shutil
print("Free disk (GB):",
      round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2))

In [ ]:
import subprocess
from pathlib import Path
import shutil

SRC = Path("/kaggle/working/CAFRNet_source")
DATA = Path("/kaggle/working/CAFRNet_continuation/cafr_data")
OUT = Path("/kaggle/working/CAFRNet_MULTI_SEED/cafrnet_seed123")

assert (SRC / "train_architecture.py").exists()
assert (DATA / "train").exists()
assert (DATA / "val").exists()
assert not OUT.exists(), "Output folder already exists. Check before starting."

free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
assert free_gb >= 2.0, f"Insufficient free disk space: {free_gb:.2f} GB"

OUT.mkdir(parents=True)

cmd = [
    "python", str(SRC / "train_architecture.py"),
    "--arch", "cafrnet_v2",
    "--data", str(DATA),
    "--out", str(OUT),
    "--epochs", "30",
    "--batch-size", "16",
    "--seed", "123",
    "--workers", "2",
    "--image-size", "224",
    "--lambda-consistency", "0.25",
    "--encoder-lr", "2e-5",
    "--head-lr", "2e-4",
    "--weight-decay", "0.05",
    "--freeze-epochs", "2",
    "--ablation", "full"
]

print("Starting CAFR-Net Full | Seed 123")
print("Output:", OUT)
subprocess.run(cmd, cwd=str(SRC), check=True)
print("\nSEED 123 TRAINING COMPLETED")

In [ ]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

keywords = [
    "oxford", "pet", "animegan", "anime",
    "stable_diffusion", "stable-diffusion",
    "cartoon", "stylized"
]

print("========== EXISTING RESEARCH ASSETS ==========")

for root in roots:
    print("\nSEARCHING:", root)

    if not root.exists():
        continue

    matches = []
    for p in root.rglob("*"):
        if any(k in p.name.lower() for k in keywords):
            matches.append(p)

        if len(matches) >= 100:
            break

    for p in matches:
        print(p)

print("\n========== END OF AUDIT ==========")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")
matches = []

for p in root.rglob("*"):
    name = p.name.lower()
    if "oxford" in name or "iiit" in name or "pet" in name:
        matches.append(p)

print("Oxford-IIIT Pet related assets:")
if matches:
    for p in matches[:100]:
        print(p)
else:
    print("No Oxford-IIIT Pet dataset found.")

In [ ]:

import requests

url = "https://raw.githubusercontent.com/Karthik6-cpu/cartoonization-robustness-study/research/cafr-net-phase-iv/Phase2_Colab_Conference_Validation_Pack%20(1).ipynb"

r = requests.get(url, timeout=30)
print("HTTP:", r.status_code)

if r.status_code == 200:
    import json
    nb = r.json()

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))
        if any(k in source.lower() for k in [
            "animegan", "stable_diffusion",
            "cartoon", "checkpoint", "prediction"
        ]):
            print(f"\n--- CELL {i} ({cell['cell_type']}) ---")
            print(source[:5000])

            outputs = cell.get("outputs", [])
            for out in outputs:
                data = out.get("data", {})
                if "text/plain" in data:
                    print("OUTPUT:", str(data["text/plain"])[:1500])
                elif "text" in out:
                    print("OUTPUT:", str(out["text"])[:1500])
else:
    print("Could not fetch notebook:", r.text[:500])


In [ ]:
import requests

url = "https://raw.githubusercontent.com"
try:
    r = requests.get(url, timeout=15)
    print("Internet connection: OK")
    print("HTTP status:", r.status_code)
except Exception as e:
    print("Internet connection: NOT AVAILABLE")
    print(type(e).__name__, str(e)[:300])

In [ ]:
from pathlib import Path
import os

ROOT = Path("/kaggle/working")

print("KAGGLE WORKSPACE INVENTORY")
print("=" * 65)

total_files = 0
total_bytes = 0

for folder in sorted(ROOT.iterdir()):
    if folder.is_dir():
        files = [p for p in folder.rglob("*") if p.is_file()]
        size = sum(p.stat().st_size for p in files)
        total_files += len(files)
        total_bytes += size

        print(f"\n📁 {folder.name}")
        print(f"   Files: {len(files)}")
        print(f"   Size:  {size / (1024**3):.3f} GB")

    elif folder.is_file():
        size = folder.stat().st_size
        total_files += 1
        total_bytes += size
        print(f"📄 {folder.name} — {size / (1024**2):.2f} MB")

print("\n" + "=" * 65)
print("TOTAL FILES:", total_files)
print("TOTAL SIZE:", round(total_bytes / (1024**3), 3), "GB")

print("\nLARGEST 25 FILES")
largest = sorted(
    (p for p in ROOT.rglob("*") if p.is_file()),
    key=lambda p: p.stat().st_size,
    reverse=True
)[:25]

for p in largest:
    print(f"{p.stat().st_size / (1024**2):9.2f} MB  {p}")

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working")

files = [p for p in ROOT.rglob("*") if p.is_file()]
files.sort(key=lambda p: p.stat().st_size, reverse=True)

print("LARGEST 40 FILES")
for p in files[:40]:
    print(f"{p.stat().st_size / (1024**2):9.2f} MB | {p}")

print("\nFILES LARGER THAN 100 MB")
large = [p for p in files if p.stat().st_size > 100 * 1024**2]
print("Count:", len(large))
for p in large:
    print(f"{p.stat().st_size / (1024**2):9.2f} MB | {p}")

print("\nFILE TYPES")
types = Counter(p.suffix.lower() or "[no extension]" for p in files)
for ext, count in types.most_common():
    print(f"{ext:18} {count:6}")

print("\nGIT / UPLOAD CHECK")
print("Existing repository folder:",
      (ROOT / "cartoonization-robustness-study").exists())
print("Total files:", len(files))
print("Total size (GB):",
      round(sum(p.stat().st_size for p in files) / (1024**3), 3))

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter

ROOT = Path("/kaggle/working")

files = [p for p in ROOT.rglob("*") if p.is_file()]

# Group files by size as a preliminary duplicate check
by_size = defaultdict(list)
for p in files:
    by_size[p.stat().st_size].append(p)

print("POTENTIAL DUPLICATES (same size, not yet hash-verified)")
for size, paths in sorted(by_size.items(), key=lambda x: -len(x[1])):
    if len(paths) > 1 and size > 1024 * 1024:
        print(f"\n{size / (1024**2):.2f} MB — {len(paths)} copies")
        for p in paths[:8]:
            print(" ", p)

print("\nIMAGE FILE LOCATIONS")
jpeg_dirs = Counter()
for p in files:
    if p.suffix.lower() in [".jpeg", ".jpg"]:
        jpeg_dirs[str(p.parent)] += 1

for folder, count in jpeg_dirs.most_common(30):
    print(f"{count:5} images | {folder}")

print("\nRESEARCH FILES BY EXTENSION")
for ext in [".py", ".ipynb", ".csv", ".json", ".jsonl", ".md", ".png"]:
    matches = [p for p in files if p.suffix.lower() == ext]
    print(f"{ext:8} {len(matches):5} files")

print("\nZIP FILES")
for p in ROOT.glob("*.zip"):
    print(f"{p.stat().st_size / (1024**3):.3f} GB | {p.name}")

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib

ROOT = Path("/kaggle/working")

by_size = defaultdict(list)

for p in ROOT.rglob("*"):
    if p.is_file() and p.stat().st_size > 1024 * 1024:
        by_size[p.stat().st_size].append(p)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("VERIFYING SAME-SIZE FILES...")
verified = defaultdict(list)

for size, paths in by_size.items():
    if len(paths) > 1:
        for p in paths:
            verified[(size, sha256(p))].append(p)

print("\nCONFIRMED IDENTICAL FILES")
duplicate_groups = 0

for (size, digest), paths in verified.items():
    if len(paths) > 1:
        duplicate_groups += 1
        print(f"\n{size / (1024**2):.2f} MB | SHA256: {digest}")
        for p in paths:
            print(" ", p)

print("\nIdentical-file groups:", duplicate_groups)

In [ ]:
import os
import subprocess

print("1. CHECKING KAGGLE SECRET")

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    token = secrets.get_secret("GITHUB_TOKEN")
    print("GITHUB_TOKEN secret: AVAILABLE" if token else "GITHUB_TOKEN secret: EMPTY")
except Exception as e:
    token = None
    print("GITHUB_TOKEN secret: NOT AVAILABLE")
    print(str(e)[:250])

print("\n2. CHECKING INTERNET ACCESS")

try:
    import requests
    response = requests.get(
        "https://api.github.com",
        timeout=15
    )
    print("GitHub API HTTP:", response.status_code)
except Exception as e:
    print("GitHub connection failed:", type(e).__name__, str(e)[:250])

print("\n3. CHECKING GIT LFS")

try:
    result = subprocess.run(
        ["git", "lfs", "version"],
        capture_output=True,
        text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
except Exception as e:
    print("Git LFS unavailable:", str(e)[:200])

print("\n4. TARGET")
print("Repository: Karthik6-cpu/cartoonization-robustness-study")
print("Branch: research/cafr-net-phase-iv")

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

print("Checking GitHub connection...")
r = requests.get("https://api.github.com", timeout=15)
print("GitHub HTTP:", r.status_code)

print("\nChecking token...")
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
r = requests.get(
    "https://api.github.com/user",
    headers={"Authorization": f"Bearer {token}"},
    timeout=15
)
print("Authentication HTTP:", r.status_code)
print("Authenticated:", r.status_code == 200)

In [ ]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
token = client.get_secret("GITHUB_TOKEN")

print("Secret successfully loaded:", bool(token))

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

BASE = "https://api.github.com/repos/Karthik6-cpu/cartoonization-robustness-study"

# Verify account
user = requests.get("https://api.github.com/user", headers=headers, timeout=20)
print("GitHub authentication:", user.status_code)

if user.status_code == 200:
    print("Authenticated account:", user.json()["login"])

# Verify repository permissions
repo = requests.get(BASE, headers=headers, timeout=20)
print("\nRepository access:", repo.status_code)

if repo.status_code == 200:
    permissions = repo.json().get("permissions", {})
    print("Push permission:", permissions.get("push"))

# Verify research branch
branch = requests.get(
    BASE + "/branches/research/cafr-net-phase-iv",
    headers=headers,
    timeout=20
)
print("\nResearch branch:", branch.status_code)

if branch.status_code == 200:
    print("Current commit:", branch.json()["commit"]["sha"])

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import shutil
import csv
import json

ROOT = Path("/kaggle/working")
STAGE = ROOT / "CAFRNet_GITHUB_STAGING"

if STAGE.exists():
    shutil.rmtree(STAGE)

STAGE.mkdir(parents=True)

MAX_GIT_FILE = 95 * 1024 * 1024  # Stay below GitHub's 100 MB limit

records = []
hash_locations = defaultdict(list)
unique_count = 0
large_count = 0
dataset_count = 0
duplicate_count = 0

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for src in ROOT.rglob("*"):
    if not src.is_file():
        continue

    if STAGE in src.parents:
        continue

    rel = src.relative_to(ROOT)

    # Dataset images are excluded from the public code repository
    if "cafr_data" in rel.parts:
        dataset_count += 1
        records.append({
            "source": str(rel),
            "status": "EXCLUDED_DATASET",
            "size_bytes": src.stat().st_size
        })
        continue

    if ".virtual_documents" in rel.parts or src.suffix == ".pyc":
        records.append({
            "source": str(rel),
            "status": "EXCLUDED_TEMPORARY",
            "size_bytes": src.stat().st_size
        })
        continue

    size = src.stat().st_size

    # Large checkpoints and archives need separate storage
    if size >= MAX_GIT_FILE:
        large_count += 1
        records.append({
            "source": str(rel),
            "status": "LARGE_REQUIRES_LFS_OR_ARTIFACT_STORAGE",
            "size_bytes": size
        })
        continue

    digest = sha256_file(src)

    if digest in hash_locations:
        duplicate_count += 1
        hash_locations[digest].append(str(rel))
        records.append({
            "source": str(rel),
            "status": "DUPLICATE_OF",
            "duplicate_of": hash_locations[digest][0],
            "sha256": digest,
            "size_bytes": size
        })
        continue

    dest = STAGE / "workspace_snapshot" / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)

    hash_locations[digest].append(str(rel))
    unique_count += 1

    records.append({
        "source": str(rel),
        "status": "STAGED",
        "staged_path": str(dest.relative_to(STAGE)),
        "sha256": digest,
        "size_bytes": size
    })

# Save complete inventory
with open(STAGE / "upload_manifest.json", "w") as f:
    json.dump(records, f, indent=2)

with open(STAGE / "upload_manifest.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=sorted({k for r in records for k in r})
    )
    writer.writeheader()
    writer.writerows(records)

staged_files = [
    p for p in STAGE.rglob("*")
    if p.is_file()
]

staged_bytes = sum(p.stat().st_size for p in staged_files)

print("STAGING COMPLETE")
print("=" * 55)
print("Unique files staged:", unique_count)
print("Duplicate copies recorded:", duplicate_count)
print("Large files reserved for separate upload:", large_count)
print("Dataset images excluded:", dataset_count)
print("Total source entries catalogued:", len(records))
print("Staged size (GB):", round(staged_bytes / (1024**3), 3))
print("Staging folder:", STAGE)
print("\nManifest files:")
print(STAGE / "upload_manifest.json")
print(STAGE / "upload_manifest.csv")

In [ ]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

ROOT = Path("/kaggle/working")
STAGE = ROOT / "CAFRNet_GITHUB_STAGING"
REPO = ROOT / "CAFRNet_GITHUB_UPLOAD_REPO"

BRANCH = "research/cafr-net-phase-iv"
EXPECTED_COMMIT = "168f69f4507e72b5417159991bdecb6bd5da2a0f"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")

assert token, "GitHub token unavailable"
assert STAGE.exists(), "Staging folder missing"

# Check staged files for accidentally included GitHub tokens
import re

secret_pattern = re.compile(
    rb"(gh[pousr]_[A-Za-z0-9_]{30,}|github_pat_[A-Za-z0-9_]{30,})"
)

suspicious = []
for p in STAGE.rglob("*"):
    if p.is_file() and p.suffix.lower() in {
        ".py", ".ipynb", ".json", ".jsonl", ".md", ".txt", ".csv"
    }:
        try:
            if secret_pattern.search(p.read_bytes()):
                suspicious.append(str(p))
        except Exception:
            pass

if suspicious:
    print("STOP: Possible credential strings found:")
    for p in suspicious:
        print(p)
    raise RuntimeError("Review these files before uploading.")

# Create temporary Git authentication helper
askpass = ROOT / "_temporary_git_askpass.sh"
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) echo "" ;;\n'
    'esac\n'
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

def run(cmd, cwd=None):
    result = subprocess.run(
        cmd, cwd=cwd, env=env,
        text=True, capture_output=True
    )
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError("Command failed: " + " ".join(cmd))
    return result.stdout.strip()

try:
    # Clone only the required branch
    if REPO.exists():
        shutil.rmtree(REPO)

    print("Cloning existing research branch...")
    run([
        "git", "clone", "--depth", "1",
        "--single-branch", "--branch", BRANCH,
        "https://github.com/Karthik6-cpu/cartoonization-robustness-study.git",
        str(REPO)
    ])

    current = run(["git", "rev-parse", "HEAD"], cwd=REPO)
    print("Cloned branch commit:", current)

    if current != EXPECTED_COMMIT:
        raise RuntimeError(
            "Branch changed since verification. Stop and re-check before pushing."
        )

    # Create a new archive directory
    destination = (
        REPO / "research" / "CAFRNet_v2" / "kaggle_workspace_archive"
    )
    destination.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        STAGE / "workspace_snapshot",
        destination / "workspace_snapshot",
        dirs_exist_ok=True
    )

    shutil.copy2(STAGE / "upload_manifest.json", destination)
    shutil.copy2(STAGE / "upload_manifest.csv", destination)

    # Add archive documentation
    (destination / "ARCHIVE_README.md").write_text(
        "# CAFR-Net Kaggle Workspace Archive\n\n"
        "This archive contains unique staged research files from the "
        "Kaggle workspace.\n\n"
        "- Dataset images are excluded.\n"
        "- Duplicate copies are recorded in the manifests.\n"
        "- Large files requiring separate artifact storage are listed "
        "in the manifest.\n"
        "- Original Kaggle workspace files were not modified.\n"
    )

    run(["git", "config", "user.name", "Karthik6-cpu"], cwd=REPO)
    run([
        "git", "config", "user.email",
        "157053962+Karthik6-cpu@users.noreply.github.com"
    ], cwd=REPO)

    run(["git", "add", "research/CAFRNet_v2/kaggle_workspace_archive"], cwd=REPO)

    print("\nGit change summary:")
    print(run(["git", "status", "--short"], cwd=REPO)[:6000])

    run([
        "git", "commit", "-m",
        "Archive unique CAFR-Net Kaggle workspace research files"
    ], cwd=REPO)

    print("\nPushing to GitHub...")
    run([
        "git", "push", "origin",
        f"HEAD:refs/heads/{BRANCH}"
    ], cwd=REPO)

    print("\nUPLOAD SUCCESSFUL")
    print("New commit:", run(["git", "rev-parse", "HEAD"], cwd=REPO))
    print("Branch:", BRANCH)
    print("Archive:", destination)

finally:
    if askpass.exists():
        askpass.unlink()
    token = None
    env.pop("GITHUB_TOKEN", None)

In [ ]:
from pathlib import Path

matches = list(Path("/kaggle/working").rglob(
    "CAFRNet_Phase_V_Complete_Research.ipynb"
))

if matches:
    for p in matches:
        print("FOUND:", p)
        print("SIZE:", round(p.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("Notebook file not found in /kaggle/working.")
    print("Save/Download the current notebook as an .ipynb file first.")